# no30 Lightning AI: nnU-Net v2 3D fullres, fold 2, seed 2026

Protocol-locked Lightning AI derivative for execution on a Tesla T4. The patient roles, fold, seed, planner, trainer, convergence rule, predictions, metrics, and detailed ZIP contract are unchanged. Dataset, preprocessing, checkpoints, and reports use persistent Studio storage.


## Install and verify the Lightning nnU-Net environment

The notebook installs the locked CUDA/PyTorch and nnU-Net package versions into the Studio environment. The actual planned model is tested with one real fit-training batch and one real inner-tuning batch before full training.


In [ ]:
# Lightning AI adapter: persistent Studio paths and exact environment.
import hashlib
import os
from pathlib import Path

LIGHTNING_STUDIO_ROOT = Path(
    os.environ.get("MU_GLIOMA_LIGHTNING_ROOT", "/teamspace/studios/this_studio")
).expanduser().resolve()
LIGHTNING_DATASET_ROOT = Path(
    os.environ.get("MU_GLIOMA_DATASET_ROOT", LIGHTNING_STUDIO_ROOT / "data")
).expanduser().resolve()
LIGHTNING_NNUNET_WORK_ROOT = Path(
    os.environ.get(
        "MU_GLIOMA_NNUNET_WORK_ROOT",
        LIGHTNING_STUDIO_ROOT / "mu_glioma_nnunet_work",
    )
).expanduser().resolve()
LIGHTNING_OUTPUT_ROOT = Path(
    os.environ.get(
        "MU_GLIOMA_OUTPUT_ROOT",
        LIGHTNING_STUDIO_ROOT / "mu_glioma_results",
    )
).expanduser().resolve()
for lightning_path in (LIGHTNING_NNUNET_WORK_ROOT, LIGHTNING_OUTPUT_ROOT):
    lightning_path.mkdir(parents=True, exist_ok=True)

import subprocess, sys
os.environ['PATH'] = str(Path(sys.executable).parent) + os.pathsep + os.environ.get('PATH', '')
# Install PyTorch first, as recommended by nnU-Net. The constraint file then
# prevents any transitive dependency from replacing the P100-compatible build.
constraints_path = str(LIGHTNING_NNUNET_WORK_ROOT / 'mu_glioma_nnunet_constraints.txt')
with open(constraints_path, 'w') as constraints_file:
    constraints_file.write(
        'torch==2.8.0+cu126\n'
        'torchvision==0.23.0+cu126\n'
        'torchaudio==2.8.0+cu126\n'
        'numpy==2.0.2\n'
        'scipy==1.14.1\n'
        'scikit-image==0.24.0\n'
        'pandas==2.2.2\n'
    )
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', '--no-cache-dir',
    'torch==2.8.0', 'torchvision==0.23.0', 'torchaudio==2.8.0',
    '--index-url', 'https://download.pytorch.org/whl/cu126'
], check=True)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', '--no-cache-dir',
    '--constraint', constraints_path,
    '--extra-index-url', 'https://download.pytorch.org/whl/cu126',
    'nnunetv2==2.8.1', 'nibabel', 'surface-distance==0.1',
    'numpy==2.0.2', 'scipy==1.14.1',
    'scikit-image==0.24.0', 'pandas==2.2.2'
], check=True)
subprocess.run(
    [sys.executable, '-m', 'pip', 'cache', 'purge'],
    check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)

import importlib.metadata
import json, os, random, shutil, tarfile, threading, time
from pathlib import Path
import nibabel as nib
import numpy as np
import pandas as pd
import scipy
import skimage
import torch
import torchvision
import torchaudio
import nnunetv2

assert torch.cuda.is_available(), (
    'No CUDA GPU is visible. Start this Lightning Studio on a CUDA GPU machine.'
)
assert torch.__version__.startswith('2.8.0+cu126'), (
    f'Expected PyTorch 2.8.0+cu126, but loaded {torch.__version__}'
)
assert torchvision.__version__.startswith('0.23.0+cu126'), (
    f'Expected torchvision 0.23.0+cu126, but loaded {torchvision.__version__}'
)
assert torchaudio.__version__.startswith('2.8.0+cu126'), (
    f'Expected torchaudio 2.8.0+cu126, but loaded {torchaudio.__version__}'
)
assert np.__version__ == '2.0.2', f'Expected NumPy 2.0.2, loaded {np.__version__}'
assert scipy.__version__ == '1.14.1', f'Expected SciPy 1.14.1, loaded {scipy.__version__}'
assert skimage.__version__ == '0.24.0', f'Expected scikit-image 0.24.0, loaded {skimage.__version__}'
assert pd.__version__ == '2.2.2', f'Expected pandas 2.2.2, loaded {pd.__version__}'
device_capability = torch.cuda.get_device_capability(0)
required_arch = f'sm_{device_capability[0]}{device_capability[1]}'
assert required_arch in torch.cuda.get_arch_list(), (
    f'This PyTorch build does not support the selected GPU ({required_arch}): '
    f'{torch.cuda.get_arch_list()}'
)

# Exercise the exact critical operation class used during 3D nnU-Net training:
# a mixed-precision Conv3D forward and backward pass on the selected GPU.
smoke_input = torch.randn((1, 1, 8, 8, 8), device='cuda', requires_grad=True)
smoke_conv = torch.nn.Conv3d(1, 2, 3, padding=1).cuda()
with torch.autocast(device_type='cuda', dtype=torch.float16):
    smoke_output = smoke_conv(smoke_input).square().mean()
smoke_output.backward()
assert torch.isfinite(smoke_output).item()
# Exercise the operator whose missing registration caused the previous late
# trainer-discovery failure. This is a package-pairing check, not model logic.
from torchvision.ops import nms
nms_result = nms(
    torch.tensor([[0., 0., 2., 2.], [0., 0., 1., 1.]]),
    torch.tensor([0.9, 0.1]), 0.5,
)
assert nms_result.tolist() == [0, 1]
del smoke_input, smoke_conv, smoke_output
torch.cuda.empty_cache()
print('GPU:', torch.cuda.get_device_name(0))
print('GPU memory:', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1), 'GiB')
print('PyTorch:', torch.__version__)
print('torchvision:', torchvision.__version__)
print('torchaudio:', torchaudio.__version__)
print('NumPy:', np.__version__)
print('SciPy:', scipy.__version__)
print('scikit-image:', skimage.__version__)
print('pandas:', pd.__version__)
print('CUDA runtime:', torch.version.cuda)
print('Supported CUDA architectures:', torch.cuda.get_arch_list())
print('Mixed-precision Conv3D forward/backward smoke test: passed')
print('torchvision compiled-operator registration test: passed')
print('nnU-Net:', Path(nnunetv2.__file__).parent)

torch_parts = torch.__version__.split('+')[0].split('.')
if tuple(map(int, torch_parts[:2])) > (2, 8):
    print('WARNING: nnU-Net reports slower 3D AMP convolutions with PyTorch 2.9+.')

DATASET_ID = 502
DATASET_NAME = 'Dataset502_MUGliomaPost'
MAX_EPOCHS = 150
MIN_EPOCHS = 100
MIN_IMPROVEMENT = 0.002
EARLY_STOPPING_PATIENCE = 15
SAVE_EVERY = 1
PREPROCESS_PROCESSES = 2
RUN_ID = 'no30'
SPLIT_SEED = 2026
INNER_SPLIT_SEED = 6204
TRAINING_SEED = 2026
CV_FOLD = 2
EXPECTED_FIT_TRAIN_CASES = 381
EXPECTED_FIT_TRAIN_PATIENTS = 130
EXPECTED_TUNING_CASES = 94
EXPECTED_TUNING_PATIENTS = 32
EXPECTED_OUTER_TEST_CASES = 119
EXPECTED_OUTER_TEST_PATIENTS = 41
EXPECTED_SPLIT_SHA256 = '675a1598256128813e41b8b301341050284feb0709de3c63bb6a72cd5601e22e'

random.seed(TRAINING_SEED)
np.random.seed(TRAINING_SEED)
torch.manual_seed(TRAINING_SEED)
torch.cuda.manual_seed_all(TRAINING_SEED)
os.environ['PYTHONHASHSEED'] = str(TRAINING_SEED)

input_root = LIGHTNING_DATASET_ROOT
tmp_root = LIGHTNING_NNUNET_WORK_ROOT / RUN_ID
raw_root = tmp_root / 'nnUNet_raw'
preprocessed_root = tmp_root / 'nnUNet_preprocessed'
output_root = LIGHTNING_OUTPUT_ROOT / 'MU_Glioma_no30_nnunet_fold2_seed2026'
results_root = output_root / 'nnUNet_results'
truth_dir = tmp_root / 'validation_truth'
outer_test_input_dir = tmp_root / 'outer_test_input'
outer_test_prediction_dir = output_root / 'outer_test_predictions'
for path in (
    raw_root, preprocessed_root, output_root, results_root, truth_dir,
    outer_test_input_dir, outer_test_prediction_dir,
):
    path.mkdir(parents=True, exist_ok=True)

os.environ['nnUNet_raw'] = str(raw_root)
os.environ['nnUNet_preprocessed'] = str(preprocessed_root)
os.environ['nnUNet_results'] = str(results_root)
os.environ['nnUNet_n_proc_DA'] = '2'
os.environ['nnUNet_compile'] = 'f'

experiment_config = {
    'archive_contract_version': 'mu_glioma_detailed_zip_v2_nested',
    'run_id': RUN_ID,
    'architecture': 'nnU-Net v2 3D fullres',
    'cv_fold': CV_FOLD,
    'split_seed': SPLIT_SEED,
    'inner_split_seed': INNER_SPLIT_SEED,
    'training_seed': TRAINING_SEED,
    'maximum_epochs': MAX_EPOCHS,
    'early_stopping_monitor': 'ema_fg_dice',
    'minimum_epochs_before_early_stopping': MIN_EPOCHS,
    'early_stopping_min_delta': MIN_IMPROVEMENT,
    'early_stopping_patience': EARLY_STOPPING_PATIENCE,
    'surface_dice_tolerance_mm': 1.0,
    'lesion_connectivity': '26-connected',
    'checkpoint_selection_role': 'patient_grouped_inner_tuning',
    'final_evaluation_role': 'untouched_outer_internal_test',
    'expected_split_sha256': EXPECTED_SPLIT_SHA256,
    'expected_fit_train_cases': EXPECTED_FIT_TRAIN_CASES,
    'expected_tuning_cases': EXPECTED_TUNING_CASES,
    'expected_outer_test_cases': EXPECTED_OUTER_TEST_CASES,
}
(output_root / 'experiment_config.json').write_text(json.dumps(experiment_config, indent=2))

environment_record = {
    'python': sys.version,
    'pytorch': torch.__version__,
    'torchvision': torchvision.__version__,
    'torchaudio': torchaudio.__version__,
    'numpy': np.__version__,
    'scipy': scipy.__version__,
    'scikit_image': skimage.__version__,
    'pandas': pd.__version__,
    'nibabel': nib.__version__,
    'nnunetv2': importlib.metadata.version('nnunetv2'),
    'surface_distance': importlib.metadata.version('surface-distance'),
    'gpu': torch.cuda.get_device_name(0),
    'cuda_runtime': torch.version.cuda,
    'execution_platform': 'lightning_ai_t4_cuda',
    'pytorch_cuda_allocator_config': os.environ.get('PYTORCH_CUDA_ALLOC_CONF'),
    'studio_root': str(LIGHTNING_STUDIO_ROOT),
    'dataset_root': str(LIGHTNING_DATASET_ROOT),
    'work_root': str(LIGHTNING_NNUNET_WORK_ROOT),
    'output_root': str(LIGHTNING_OUTPUT_ROOT),
}
(output_root / 'environment.json').write_text(json.dumps(environment_record, indent=2))

TRAINER_NAME = 'nnUNetTrainer_MUGlioma_no30'
trainer_file = (Path(nnunetv2.__file__).parent / 'training' / 'nnUNetTrainer' /
                'variants' / 'training_length' / f'{TRAINER_NAME}.py')
trainer_file.write_text("""import json
import os
import random
import numpy as np
import torch
from nnunetv2.training.nnUNetTrainer.nnUNetTrainer import nnUNetTrainer

class nnUNetTrainer_MUGlioma_no30(nnUNetTrainer):
    # Exact base signature is required by nnU-Net trainer introspection.
    def __init__(
        self,
        plans: dict,
        configuration: str,
        fold: int,
        dataset_json: dict,
        device: torch.device = torch.device('cuda'),
    ):
        random.seed(2026)
        np.random.seed(2026)
        torch.manual_seed(2026)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(2026)
        super().__init__(plans, configuration, fold, dataset_json, device)
        self.num_epochs = 150
        self.save_every = 1
        self.early_stopping_min_epochs = 100
        self.early_stopping_min_delta = 0.002
        self.early_stopping_patience = 15

    def initialize(self):
        random.seed(2026)
        np.random.seed(2026)
        torch.manual_seed(2026)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(2026)
        return super().initialize()

    def _convergence_state(self):
        history = [
            float(value)
            for value in self.logger.get_value('ema_fg_dice', step=None)
        ]
        best_value = None
        best_epoch = None
        epochs_without_improvement = 0
        for epoch, value in enumerate(history, start=1):
            if np.isfinite(value) and (
                best_value is None
                or value > best_value + self.early_stopping_min_delta
            ):
                best_value = value
                best_epoch = epoch
                epochs_without_improvement = 0
            else:
                epochs_without_improvement += 1
        return {
            'monitor': 'ema_fg_dice',
            'completed_epochs': len(history),
            'minimum_epochs_before_early_stopping': self.early_stopping_min_epochs,
            'min_improvement': self.early_stopping_min_delta,
            'patience': self.early_stopping_patience,
            'meaningful_best_epoch': best_epoch,
            'meaningful_best_ema_fg_dice': best_value,
            'epochs_without_meaningful_improvement': epochs_without_improvement,
            'should_stop': (
                len(history) >= self.early_stopping_min_epochs
                and epochs_without_improvement >= self.early_stopping_patience
            ),
        }

    def _write_convergence_state(self, state):
        with open(
            os.path.join(self.output_folder, 'convergence_state.json'),
            'w', encoding='utf-8'
        ) as handle:
            json.dump(state, handle, indent=2)

    def run_training(self):
        self.on_train_start()
        state = self._convergence_state()
        stop_reason = None

        if state['should_stop']:
            stop_reason = 'early_stopping_after_resume'
        else:
            for _ in range(self.current_epoch, self.num_epochs):
                self.on_epoch_start()
                self.on_train_epoch_start()
                train_outputs = []
                for _ in range(self.num_iterations_per_epoch):
                    train_outputs.append(self.train_step(next(self.dataloader_train)))
                self.on_train_epoch_end(train_outputs)

                with torch.no_grad():
                    self.on_validation_epoch_start()
                    val_outputs = []
                    for _ in range(self.num_val_iterations_per_epoch):
                        val_outputs.append(self.validation_step(next(self.dataloader_val)))
                    self.on_validation_epoch_end(val_outputs)

                self.on_epoch_end()
                state = self._convergence_state()
                self._write_convergence_state(state)
                if state['should_stop']:
                    stop_reason = 'early_stopping'
                    break

        if stop_reason is None:
            stop_reason = 'maximum_epochs'
        state = self._convergence_state()
        state['stop_reason'] = stop_reason
        state['maximum_epochs'] = self.num_epochs
        self._write_convergence_state(state)
        self.print_to_log_file(
            'Training stop reason:', stop_reason,
            '| completed epochs:', state['completed_epochs'],
            '| meaningful best epoch:', state['meaningful_best_epoch'],
            '| epochs without meaningful improvement:',
            state['epochs_without_meaningful_improvement'],
        )
        self.on_train_end()
""")
from nnunetv2.utilities.find_objects import recursive_find_trainer_class_by_name
discovered_trainer = recursive_find_trainer_class_by_name(TRAINER_NAME)
assert discovered_trainer is not None, f'Could not discover custom trainer {TRAINER_NAME}'
assert discovered_trainer.__name__ == TRAINER_NAME
print('Full nnU-Net trainer discovery/import test: passed')
print('Trainer:', TRAINER_NAME)
print('Persistent output:', output_root)


## Locate the persistent Lightning dataset

The adapter reads the dataset under `/teamspace/studios/this_studio/data`. Generated nnU-Net raw links, preprocessing, checkpoints, and predictions use separate persistent Studio directories.


In [ ]:
def find_source(root):
    patient_dirs = [p for p in root.rglob('PatientID_*') if p.is_dir()]
    if not patient_dirs:
        return None
    candidates = {}
    for patient_dir in patient_dirs:
        parent = patient_dir.parent
        candidates[parent] = candidates.get(parent, 0) + 1
    return max(candidates, key=candidates.get)

source = find_source(input_root)
extracted_source_root = None

if source is None:
    archives = list(input_root.rglob('MU-Glioma-Post-Kaggle.bundle'))
    if not archives:
        archives = list(input_root.rglob('*.bundle'))
    if not archives:
        archives = list(input_root.rglob('MU-Glioma-Post-Kaggle.tar'))
    if not archives:
        archives = list(input_root.rglob('*.tar'))
    assert archives, (
        'Dataset archive not found. Add the private Kaggle Dataset containing '
        'MU-Glioma-Post-Kaggle.bundle as notebook input.'
    )
    archive = archives[0]
    extracted_source_root = tmp_root / 'uploaded_source'
    marker = extracted_source_root / '.extraction_complete'

    def extracted_bundle_is_complete():
        if not marker.exists():
            return False
        required_suffixes = ('brain_t1n', 'brain_t1c', 'brain_t2w', 'brain_t2f')
        return all(
            len(list(extracted_source_root.rglob(f'*_{suffix}.nii.gz'))) == 596
            for suffix in required_suffixes
        ) and len(list(extracted_source_root.rglob('*_tumorMask.nii.gz'))) == 594

    if not extracted_bundle_is_complete():
        if extracted_source_root.exists():
            shutil.rmtree(extracted_source_root)
        extracted_source_root.mkdir(parents=True)
        subprocess.run(
            ['tar', '-tf', str(archive)], check=True,
            stdout=subprocess.DEVNULL,
        )
        print('Uploaded bundle integrity check: passed')
        print('Extracting the 12 GB archive. This may take several minutes:', archive)
        subprocess.run(['tar', '-xf', str(archive), '-C', str(extracted_source_root)], check=True)
        marker.touch()
        assert extracted_bundle_is_complete(), 'Bundle extraction did not produce the expected 596 scans / 594 masks.'
    source = find_source(extracted_source_root)

assert source is not None and any(source.glob('PatientID_*'))
print('Resolved source:', source)
disk = shutil.disk_usage(tmp_root)
print(f'Temporary disk: {disk.used/2**30:.1f} GiB used / {disk.total/2**30:.1f} GiB total')
assert disk.free >= 28 * 2**30, (
    f'Only {disk.free/2**30:.1f} GiB temporary space remains after extraction; '
    'at least 28 GiB is required before preprocessing. Stop instead of risking a late disk failure.'
)


## Reconstruct and verify patient fold 2

The same seed-2026 balanced patient-level outer folds and the same
locked inner-tuning split are used across architectures. This notebook uses
training seed 2026; partition membership does not change with
the training seed. The outer test is excluded from nnU-Net fingerprinting,
planning, training, tuning, early stopping, and checkpoint selection.


In [ ]:
modalities = [
    ('brain_t1n', '0000'),
    ('brain_t1c', '0001'),
    ('brain_t2w', '0002'),
    ('brain_t2f', '0003'),
]

cases = []
missing_masks = []
for patient_dir in sorted(source.glob('PatientID_*')):
    for tp_dir in sorted(patient_dir.glob('Timepoint_*')):
        case_id = f'{patient_dir.name}_{tp_dir.name}'
        mask = tp_dir / f'{case_id}_tumorMask.nii.gz'
        images = [tp_dir / f'{case_id}_{name}.nii.gz' for name, _ in modalities]
        if mask.exists() and all(path.exists() for path in images):
            cases.append({
                'id': case_id,
                'patient': patient_dir.name,
                'images': images,
                'mask': mask,
            })
        elif all(path.exists() for path in images):
            missing_masks.append(case_id)

assert len(cases) == 594, f'Expected 594 complete labelled cases, found {len(cases)}'
case_by_id = {case['id']: case for case in cases}
patients = sorted({case['patient'] for case in cases})
split_record = json.loads(r"""{
  "design": "five_outer_folds_with_patient_grouped_inner_tuning_v2",
  "outer_fold": 2,
  "outer_split_seed": 2026,
  "inner_split_seed": 6204,
  "inner_tuning_fraction_of_outer_training_patients": 0.2,
  "fit_train_patients": [
    "PatientID_0004",
    "PatientID_0007",
    "PatientID_0008",
    "PatientID_0010",
    "PatientID_0011",
    "PatientID_0012",
    "PatientID_0014",
    "PatientID_0019",
    "PatientID_0021",
    "PatientID_0022",
    "PatientID_0024",
    "PatientID_0025",
    "PatientID_0026",
    "PatientID_0027",
    "PatientID_0029",
    "PatientID_0030",
    "PatientID_0031",
    "PatientID_0032",
    "PatientID_0033",
    "PatientID_0036",
    "PatientID_0037",
    "PatientID_0038",
    "PatientID_0039",
    "PatientID_0042",
    "PatientID_0043",
    "PatientID_0044",
    "PatientID_0047",
    "PatientID_0049",
    "PatientID_0051",
    "PatientID_0053",
    "PatientID_0054",
    "PatientID_0056",
    "PatientID_0061",
    "PatientID_0063",
    "PatientID_0066",
    "PatientID_0067",
    "PatientID_0068",
    "PatientID_0069",
    "PatientID_0071",
    "PatientID_0076",
    "PatientID_0077",
    "PatientID_0078",
    "PatientID_0079",
    "PatientID_0081",
    "PatientID_0083",
    "PatientID_0085",
    "PatientID_0086",
    "PatientID_0087",
    "PatientID_0089",
    "PatientID_0090",
    "PatientID_0091",
    "PatientID_0092",
    "PatientID_0093",
    "PatientID_0094",
    "PatientID_0097",
    "PatientID_0099",
    "PatientID_0101",
    "PatientID_0103",
    "PatientID_0105",
    "PatientID_0106",
    "PatientID_0108",
    "PatientID_0109",
    "PatientID_0110",
    "PatientID_0111",
    "PatientID_0113",
    "PatientID_0115",
    "PatientID_0118",
    "PatientID_0121",
    "PatientID_0123",
    "PatientID_0125",
    "PatientID_0126",
    "PatientID_0130",
    "PatientID_0133",
    "PatientID_0141",
    "PatientID_0142",
    "PatientID_0143",
    "PatientID_0145",
    "PatientID_0149",
    "PatientID_0150",
    "PatientID_0151",
    "PatientID_0152",
    "PatientID_0155",
    "PatientID_0156",
    "PatientID_0159",
    "PatientID_0160",
    "PatientID_0161",
    "PatientID_0164",
    "PatientID_0165",
    "PatientID_0168",
    "PatientID_0169",
    "PatientID_0170",
    "PatientID_0171",
    "PatientID_0173",
    "PatientID_0186",
    "PatientID_0187",
    "PatientID_0195",
    "PatientID_0197",
    "PatientID_0198",
    "PatientID_0199",
    "PatientID_0200",
    "PatientID_0201",
    "PatientID_0205",
    "PatientID_0208",
    "PatientID_0209",
    "PatientID_0210",
    "PatientID_0212",
    "PatientID_0213",
    "PatientID_0214",
    "PatientID_0216",
    "PatientID_0220",
    "PatientID_0237",
    "PatientID_0238",
    "PatientID_0239",
    "PatientID_0240",
    "PatientID_0242",
    "PatientID_0249",
    "PatientID_0250",
    "PatientID_0255",
    "PatientID_0256",
    "PatientID_0258",
    "PatientID_0259",
    "PatientID_0260",
    "PatientID_0261",
    "PatientID_0264",
    "PatientID_0265",
    "PatientID_0267",
    "PatientID_0270",
    "PatientID_0272",
    "PatientID_0273",
    "PatientID_0275"
  ],
  "tuning_patients": [
    "PatientID_0005",
    "PatientID_0006",
    "PatientID_0013",
    "PatientID_0015",
    "PatientID_0018",
    "PatientID_0034",
    "PatientID_0040",
    "PatientID_0041",
    "PatientID_0045",
    "PatientID_0055",
    "PatientID_0058",
    "PatientID_0060",
    "PatientID_0064",
    "PatientID_0074",
    "PatientID_0080",
    "PatientID_0082",
    "PatientID_0084",
    "PatientID_0116",
    "PatientID_0120",
    "PatientID_0144",
    "PatientID_0147",
    "PatientID_0148",
    "PatientID_0157",
    "PatientID_0158",
    "PatientID_0174",
    "PatientID_0188",
    "PatientID_0196",
    "PatientID_0207",
    "PatientID_0235",
    "PatientID_0236",
    "PatientID_0268",
    "PatientID_0274"
  ],
  "outer_test_patients": [
    "PatientID_0003",
    "PatientID_0009",
    "PatientID_0020",
    "PatientID_0023",
    "PatientID_0035",
    "PatientID_0046",
    "PatientID_0052",
    "PatientID_0057",
    "PatientID_0059",
    "PatientID_0062",
    "PatientID_0065",
    "PatientID_0070",
    "PatientID_0072",
    "PatientID_0073",
    "PatientID_0088",
    "PatientID_0095",
    "PatientID_0112",
    "PatientID_0114",
    "PatientID_0117",
    "PatientID_0124",
    "PatientID_0132",
    "PatientID_0135",
    "PatientID_0136",
    "PatientID_0138",
    "PatientID_0139",
    "PatientID_0140",
    "PatientID_0162",
    "PatientID_0189",
    "PatientID_0191",
    "PatientID_0192",
    "PatientID_0202",
    "PatientID_0203",
    "PatientID_0204",
    "PatientID_0206",
    "PatientID_0211",
    "PatientID_0215",
    "PatientID_0244",
    "PatientID_0251",
    "PatientID_0252",
    "PatientID_0254",
    "PatientID_0271"
  ],
  "fit_train_cases": [
    "PatientID_0004_Timepoint_1",
    "PatientID_0007_Timepoint_2",
    "PatientID_0007_Timepoint_3",
    "PatientID_0007_Timepoint_4",
    "PatientID_0007_Timepoint_5",
    "PatientID_0007_Timepoint_6",
    "PatientID_0008_Timepoint_4",
    "PatientID_0008_Timepoint_6",
    "PatientID_0010_Timepoint_1",
    "PatientID_0010_Timepoint_4",
    "PatientID_0010_Timepoint_5",
    "PatientID_0011_Timepoint_1",
    "PatientID_0011_Timepoint_2",
    "PatientID_0012_Timepoint_2",
    "PatientID_0012_Timepoint_3",
    "PatientID_0014_Timepoint_1",
    "PatientID_0014_Timepoint_2",
    "PatientID_0014_Timepoint_3",
    "PatientID_0014_Timepoint_4",
    "PatientID_0014_Timepoint_5",
    "PatientID_0014_Timepoint_6",
    "PatientID_0019_Timepoint_4",
    "PatientID_0019_Timepoint_5",
    "PatientID_0019_Timepoint_6",
    "PatientID_0021_Timepoint_2",
    "PatientID_0021_Timepoint_3",
    "PatientID_0021_Timepoint_4",
    "PatientID_0021_Timepoint_5",
    "PatientID_0021_Timepoint_6",
    "PatientID_0022_Timepoint_1",
    "PatientID_0022_Timepoint_2",
    "PatientID_0022_Timepoint_3",
    "PatientID_0024_Timepoint_2",
    "PatientID_0024_Timepoint_3",
    "PatientID_0025_Timepoint_1",
    "PatientID_0025_Timepoint_2",
    "PatientID_0025_Timepoint_3",
    "PatientID_0026_Timepoint_1",
    "PatientID_0026_Timepoint_2",
    "PatientID_0027_Timepoint_1",
    "PatientID_0029_Timepoint_1",
    "PatientID_0029_Timepoint_3",
    "PatientID_0029_Timepoint_4",
    "PatientID_0030_Timepoint_1",
    "PatientID_0030_Timepoint_3",
    "PatientID_0030_Timepoint_4",
    "PatientID_0031_Timepoint_2",
    "PatientID_0031_Timepoint_3",
    "PatientID_0031_Timepoint_4",
    "PatientID_0031_Timepoint_5",
    "PatientID_0032_Timepoint_1",
    "PatientID_0032_Timepoint_2",
    "PatientID_0032_Timepoint_3",
    "PatientID_0032_Timepoint_4",
    "PatientID_0032_Timepoint_5",
    "PatientID_0032_Timepoint_6",
    "PatientID_0033_Timepoint_1",
    "PatientID_0033_Timepoint_2",
    "PatientID_0033_Timepoint_3",
    "PatientID_0033_Timepoint_4",
    "PatientID_0033_Timepoint_5",
    "PatientID_0036_Timepoint_1",
    "PatientID_0036_Timepoint_2",
    "PatientID_0036_Timepoint_3",
    "PatientID_0036_Timepoint_4",
    "PatientID_0036_Timepoint_5",
    "PatientID_0036_Timepoint_6",
    "PatientID_0037_Timepoint_1",
    "PatientID_0037_Timepoint_2",
    "PatientID_0037_Timepoint_3",
    "PatientID_0037_Timepoint_4",
    "PatientID_0038_Timepoint_1",
    "PatientID_0038_Timepoint_2",
    "PatientID_0038_Timepoint_3",
    "PatientID_0038_Timepoint_4",
    "PatientID_0038_Timepoint_5",
    "PatientID_0038_Timepoint_6",
    "PatientID_0039_Timepoint_1",
    "PatientID_0039_Timepoint_2",
    "PatientID_0039_Timepoint_3",
    "PatientID_0039_Timepoint_4",
    "PatientID_0039_Timepoint_6",
    "PatientID_0042_Timepoint_1",
    "PatientID_0043_Timepoint_1",
    "PatientID_0044_Timepoint_1",
    "PatientID_0044_Timepoint_2",
    "PatientID_0044_Timepoint_3",
    "PatientID_0044_Timepoint_4",
    "PatientID_0044_Timepoint_5",
    "PatientID_0044_Timepoint_6",
    "PatientID_0047_Timepoint_1",
    "PatientID_0049_Timepoint_1",
    "PatientID_0051_Timepoint_1",
    "PatientID_0051_Timepoint_3",
    "PatientID_0051_Timepoint_4",
    "PatientID_0053_Timepoint_1",
    "PatientID_0053_Timepoint_3",
    "PatientID_0053_Timepoint_4",
    "PatientID_0053_Timepoint_5",
    "PatientID_0053_Timepoint_6",
    "PatientID_0054_Timepoint_1",
    "PatientID_0054_Timepoint_2",
    "PatientID_0056_Timepoint_2",
    "PatientID_0061_Timepoint_2",
    "PatientID_0063_Timepoint_1",
    "PatientID_0066_Timepoint_1",
    "PatientID_0066_Timepoint_2",
    "PatientID_0066_Timepoint_3",
    "PatientID_0066_Timepoint_5",
    "PatientID_0066_Timepoint_6",
    "PatientID_0067_Timepoint_1",
    "PatientID_0067_Timepoint_2",
    "PatientID_0068_Timepoint_1",
    "PatientID_0068_Timepoint_2",
    "PatientID_0068_Timepoint_3",
    "PatientID_0068_Timepoint_4",
    "PatientID_0068_Timepoint_5",
    "PatientID_0069_Timepoint_2",
    "PatientID_0069_Timepoint_3",
    "PatientID_0069_Timepoint_4",
    "PatientID_0069_Timepoint_5",
    "PatientID_0069_Timepoint_6",
    "PatientID_0071_Timepoint_1",
    "PatientID_0076_Timepoint_1",
    "PatientID_0076_Timepoint_2",
    "PatientID_0077_Timepoint_1",
    "PatientID_0078_Timepoint_1",
    "PatientID_0078_Timepoint_2",
    "PatientID_0078_Timepoint_3",
    "PatientID_0079_Timepoint_1",
    "PatientID_0079_Timepoint_2",
    "PatientID_0079_Timepoint_3",
    "PatientID_0079_Timepoint_4",
    "PatientID_0079_Timepoint_5",
    "PatientID_0079_Timepoint_6",
    "PatientID_0081_Timepoint_1",
    "PatientID_0081_Timepoint_2",
    "PatientID_0083_Timepoint_1",
    "PatientID_0083_Timepoint_2",
    "PatientID_0083_Timepoint_3",
    "PatientID_0083_Timepoint_4",
    "PatientID_0083_Timepoint_5",
    "PatientID_0083_Timepoint_6",
    "PatientID_0085_Timepoint_1",
    "PatientID_0085_Timepoint_2",
    "PatientID_0085_Timepoint_3",
    "PatientID_0085_Timepoint_4",
    "PatientID_0086_Timepoint_1",
    "PatientID_0087_Timepoint_1",
    "PatientID_0087_Timepoint_2",
    "PatientID_0087_Timepoint_3",
    "PatientID_0089_Timepoint_1",
    "PatientID_0089_Timepoint_2",
    "PatientID_0089_Timepoint_3",
    "PatientID_0089_Timepoint_4",
    "PatientID_0089_Timepoint_5",
    "PatientID_0089_Timepoint_6",
    "PatientID_0090_Timepoint_1",
    "PatientID_0090_Timepoint_2",
    "PatientID_0091_Timepoint_1",
    "PatientID_0091_Timepoint_2",
    "PatientID_0092_Timepoint_1",
    "PatientID_0092_Timepoint_2",
    "PatientID_0093_Timepoint_1",
    "PatientID_0093_Timepoint_2",
    "PatientID_0093_Timepoint_3",
    "PatientID_0093_Timepoint_4",
    "PatientID_0094_Timepoint_1",
    "PatientID_0094_Timepoint_2",
    "PatientID_0094_Timepoint_3",
    "PatientID_0094_Timepoint_4",
    "PatientID_0094_Timepoint_5",
    "PatientID_0094_Timepoint_6",
    "PatientID_0097_Timepoint_1",
    "PatientID_0097_Timepoint_2",
    "PatientID_0097_Timepoint_3",
    "PatientID_0097_Timepoint_4",
    "PatientID_0097_Timepoint_5",
    "PatientID_0097_Timepoint_6",
    "PatientID_0099_Timepoint_1",
    "PatientID_0099_Timepoint_2",
    "PatientID_0101_Timepoint_1",
    "PatientID_0103_Timepoint_1",
    "PatientID_0103_Timepoint_2",
    "PatientID_0103_Timepoint_3",
    "PatientID_0103_Timepoint_4",
    "PatientID_0103_Timepoint_5",
    "PatientID_0103_Timepoint_6",
    "PatientID_0105_Timepoint_1",
    "PatientID_0105_Timepoint_2",
    "PatientID_0105_Timepoint_3",
    "PatientID_0105_Timepoint_4",
    "PatientID_0105_Timepoint_5",
    "PatientID_0105_Timepoint_6",
    "PatientID_0106_Timepoint_2",
    "PatientID_0108_Timepoint_1",
    "PatientID_0109_Timepoint_1",
    "PatientID_0109_Timepoint_2",
    "PatientID_0109_Timepoint_3",
    "PatientID_0109_Timepoint_4",
    "PatientID_0110_Timepoint_1",
    "PatientID_0110_Timepoint_2",
    "PatientID_0110_Timepoint_3",
    "PatientID_0111_Timepoint_1",
    "PatientID_0113_Timepoint_1",
    "PatientID_0113_Timepoint_2",
    "PatientID_0113_Timepoint_3",
    "PatientID_0115_Timepoint_1",
    "PatientID_0118_Timepoint_1",
    "PatientID_0118_Timepoint_2",
    "PatientID_0118_Timepoint_3",
    "PatientID_0121_Timepoint_1",
    "PatientID_0121_Timepoint_2",
    "PatientID_0121_Timepoint_3",
    "PatientID_0123_Timepoint_1",
    "PatientID_0123_Timepoint_2",
    "PatientID_0125_Timepoint_1",
    "PatientID_0125_Timepoint_2",
    "PatientID_0126_Timepoint_1",
    "PatientID_0130_Timepoint_1",
    "PatientID_0133_Timepoint_1",
    "PatientID_0141_Timepoint_1",
    "PatientID_0142_Timepoint_1",
    "PatientID_0142_Timepoint_2",
    "PatientID_0143_Timepoint_1",
    "PatientID_0145_Timepoint_1",
    "PatientID_0145_Timepoint_2",
    "PatientID_0149_Timepoint_1",
    "PatientID_0149_Timepoint_2",
    "PatientID_0149_Timepoint_3",
    "PatientID_0149_Timepoint_4",
    "PatientID_0150_Timepoint_1",
    "PatientID_0150_Timepoint_2",
    "PatientID_0150_Timepoint_3",
    "PatientID_0150_Timepoint_4",
    "PatientID_0150_Timepoint_6",
    "PatientID_0151_Timepoint_1",
    "PatientID_0151_Timepoint_2",
    "PatientID_0152_Timepoint_1",
    "PatientID_0152_Timepoint_2",
    "PatientID_0152_Timepoint_3",
    "PatientID_0155_Timepoint_1",
    "PatientID_0156_Timepoint_1",
    "PatientID_0156_Timepoint_2",
    "PatientID_0156_Timepoint_3",
    "PatientID_0159_Timepoint_1",
    "PatientID_0159_Timepoint_2",
    "PatientID_0159_Timepoint_3",
    "PatientID_0159_Timepoint_4",
    "PatientID_0160_Timepoint_1",
    "PatientID_0160_Timepoint_2",
    "PatientID_0161_Timepoint_1",
    "PatientID_0164_Timepoint_1",
    "PatientID_0164_Timepoint_2",
    "PatientID_0164_Timepoint_3",
    "PatientID_0164_Timepoint_4",
    "PatientID_0164_Timepoint_5",
    "PatientID_0165_Timepoint_1",
    "PatientID_0168_Timepoint_1",
    "PatientID_0169_Timepoint_1",
    "PatientID_0169_Timepoint_2",
    "PatientID_0169_Timepoint_3",
    "PatientID_0169_Timepoint_4",
    "PatientID_0170_Timepoint_1",
    "PatientID_0170_Timepoint_2",
    "PatientID_0171_Timepoint_1",
    "PatientID_0171_Timepoint_2",
    "PatientID_0171_Timepoint_3",
    "PatientID_0173_Timepoint_1",
    "PatientID_0173_Timepoint_2",
    "PatientID_0186_Timepoint_1",
    "PatientID_0186_Timepoint_2",
    "PatientID_0186_Timepoint_3",
    "PatientID_0186_Timepoint_4",
    "PatientID_0186_Timepoint_5",
    "PatientID_0186_Timepoint_6",
    "PatientID_0187_Timepoint_1",
    "PatientID_0187_Timepoint_2",
    "PatientID_0195_Timepoint_1",
    "PatientID_0195_Timepoint_2",
    "PatientID_0195_Timepoint_3",
    "PatientID_0197_Timepoint_1",
    "PatientID_0197_Timepoint_2",
    "PatientID_0197_Timepoint_3",
    "PatientID_0198_Timepoint_1",
    "PatientID_0198_Timepoint_2",
    "PatientID_0198_Timepoint_3",
    "PatientID_0198_Timepoint_4",
    "PatientID_0199_Timepoint_1",
    "PatientID_0199_Timepoint_2",
    "PatientID_0199_Timepoint_3",
    "PatientID_0200_Timepoint_1",
    "PatientID_0201_Timepoint_1",
    "PatientID_0201_Timepoint_2",
    "PatientID_0201_Timepoint_3",
    "PatientID_0201_Timepoint_4",
    "PatientID_0201_Timepoint_5",
    "PatientID_0201_Timepoint_6",
    "PatientID_0205_Timepoint_1",
    "PatientID_0205_Timepoint_2",
    "PatientID_0205_Timepoint_3",
    "PatientID_0205_Timepoint_4",
    "PatientID_0205_Timepoint_5",
    "PatientID_0208_Timepoint_1",
    "PatientID_0208_Timepoint_2",
    "PatientID_0209_Timepoint_1",
    "PatientID_0209_Timepoint_2",
    "PatientID_0209_Timepoint_3",
    "PatientID_0209_Timepoint_4",
    "PatientID_0209_Timepoint_5",
    "PatientID_0209_Timepoint_6",
    "PatientID_0210_Timepoint_1",
    "PatientID_0210_Timepoint_2",
    "PatientID_0210_Timepoint_3",
    "PatientID_0210_Timepoint_4",
    "PatientID_0210_Timepoint_6",
    "PatientID_0212_Timepoint_1",
    "PatientID_0212_Timepoint_2",
    "PatientID_0212_Timepoint_3",
    "PatientID_0213_Timepoint_1",
    "PatientID_0213_Timepoint_2",
    "PatientID_0213_Timepoint_3",
    "PatientID_0213_Timepoint_4",
    "PatientID_0213_Timepoint_5",
    "PatientID_0214_Timepoint_1",
    "PatientID_0214_Timepoint_2",
    "PatientID_0214_Timepoint_3",
    "PatientID_0214_Timepoint_4",
    "PatientID_0214_Timepoint_5",
    "PatientID_0214_Timepoint_6",
    "PatientID_0216_Timepoint_1",
    "PatientID_0220_Timepoint_1",
    "PatientID_0220_Timepoint_6",
    "PatientID_0237_Timepoint_1",
    "PatientID_0237_Timepoint_3",
    "PatientID_0238_Timepoint_1",
    "PatientID_0238_Timepoint_3",
    "PatientID_0238_Timepoint_6",
    "PatientID_0239_Timepoint_1",
    "PatientID_0239_Timepoint_3",
    "PatientID_0239_Timepoint_6",
    "PatientID_0240_Timepoint_1",
    "PatientID_0240_Timepoint_3",
    "PatientID_0240_Timepoint_6",
    "PatientID_0242_Timepoint_1",
    "PatientID_0242_Timepoint_3",
    "PatientID_0242_Timepoint_6",
    "PatientID_0249_Timepoint_1",
    "PatientID_0249_Timepoint_3",
    "PatientID_0250_Timepoint_1",
    "PatientID_0250_Timepoint_2",
    "PatientID_0250_Timepoint_3",
    "PatientID_0255_Timepoint_1",
    "PatientID_0255_Timepoint_2",
    "PatientID_0256_Timepoint_1",
    "PatientID_0256_Timepoint_2",
    "PatientID_0256_Timepoint_3",
    "PatientID_0256_Timepoint_4",
    "PatientID_0256_Timepoint_5",
    "PatientID_0258_Timepoint_1",
    "PatientID_0259_Timepoint_1",
    "PatientID_0259_Timepoint_2",
    "PatientID_0260_Timepoint_1",
    "PatientID_0260_Timepoint_2",
    "PatientID_0261_Timepoint_1",
    "PatientID_0264_Timepoint_1",
    "PatientID_0264_Timepoint_2",
    "PatientID_0265_Timepoint_1",
    "PatientID_0265_Timepoint_2",
    "PatientID_0267_Timepoint_1",
    "PatientID_0267_Timepoint_2",
    "PatientID_0270_Timepoint_1",
    "PatientID_0270_Timepoint_3",
    "PatientID_0270_Timepoint_6",
    "PatientID_0272_Timepoint_1",
    "PatientID_0272_Timepoint_3",
    "PatientID_0272_Timepoint_6",
    "PatientID_0273_Timepoint_1",
    "PatientID_0275_Timepoint_1",
    "PatientID_0275_Timepoint_3",
    "PatientID_0275_Timepoint_6"
  ],
  "tuning_cases": [
    "PatientID_0005_Timepoint_3",
    "PatientID_0005_Timepoint_4",
    "PatientID_0006_Timepoint_2",
    "PatientID_0006_Timepoint_4",
    "PatientID_0006_Timepoint_5",
    "PatientID_0006_Timepoint_6",
    "PatientID_0013_Timepoint_1",
    "PatientID_0013_Timepoint_2",
    "PatientID_0013_Timepoint_4",
    "PatientID_0013_Timepoint_5",
    "PatientID_0013_Timepoint_6",
    "PatientID_0015_Timepoint_1",
    "PatientID_0018_Timepoint_1",
    "PatientID_0018_Timepoint_2",
    "PatientID_0018_Timepoint_4",
    "PatientID_0018_Timepoint_5",
    "PatientID_0018_Timepoint_6",
    "PatientID_0034_Timepoint_1",
    "PatientID_0034_Timepoint_2",
    "PatientID_0034_Timepoint_3",
    "PatientID_0034_Timepoint_4",
    "PatientID_0034_Timepoint_5",
    "PatientID_0034_Timepoint_6",
    "PatientID_0040_Timepoint_1",
    "PatientID_0041_Timepoint_1",
    "PatientID_0041_Timepoint_2",
    "PatientID_0045_Timepoint_1",
    "PatientID_0045_Timepoint_2",
    "PatientID_0045_Timepoint_3",
    "PatientID_0045_Timepoint_4",
    "PatientID_0055_Timepoint_1",
    "PatientID_0055_Timepoint_4",
    "PatientID_0055_Timepoint_5",
    "PatientID_0055_Timepoint_6",
    "PatientID_0058_Timepoint_1",
    "PatientID_0060_Timepoint_1",
    "PatientID_0060_Timepoint_2",
    "PatientID_0064_Timepoint_1",
    "PatientID_0064_Timepoint_2",
    "PatientID_0074_Timepoint_1",
    "PatientID_0074_Timepoint_2",
    "PatientID_0074_Timepoint_3",
    "PatientID_0074_Timepoint_4",
    "PatientID_0074_Timepoint_5",
    "PatientID_0074_Timepoint_6",
    "PatientID_0080_Timepoint_1",
    "PatientID_0080_Timepoint_2",
    "PatientID_0082_Timepoint_1",
    "PatientID_0082_Timepoint_2",
    "PatientID_0084_Timepoint_1",
    "PatientID_0084_Timepoint_2",
    "PatientID_0084_Timepoint_3",
    "PatientID_0084_Timepoint_4",
    "PatientID_0084_Timepoint_5",
    "PatientID_0084_Timepoint_6",
    "PatientID_0116_Timepoint_1",
    "PatientID_0116_Timepoint_2",
    "PatientID_0120_Timepoint_1",
    "PatientID_0144_Timepoint_1",
    "PatientID_0147_Timepoint_1",
    "PatientID_0147_Timepoint_2",
    "PatientID_0147_Timepoint_3",
    "PatientID_0148_Timepoint_1",
    "PatientID_0148_Timepoint_2",
    "PatientID_0157_Timepoint_1",
    "PatientID_0157_Timepoint_2",
    "PatientID_0157_Timepoint_3",
    "PatientID_0157_Timepoint_4",
    "PatientID_0157_Timepoint_5",
    "PatientID_0157_Timepoint_6",
    "PatientID_0158_Timepoint_1",
    "PatientID_0174_Timepoint_1",
    "PatientID_0188_Timepoint_1",
    "PatientID_0188_Timepoint_2",
    "PatientID_0196_Timepoint_1",
    "PatientID_0196_Timepoint_2",
    "PatientID_0196_Timepoint_3",
    "PatientID_0207_Timepoint_1",
    "PatientID_0207_Timepoint_2",
    "PatientID_0207_Timepoint_3",
    "PatientID_0207_Timepoint_4",
    "PatientID_0207_Timepoint_5",
    "PatientID_0207_Timepoint_6",
    "PatientID_0235_Timepoint_1",
    "PatientID_0235_Timepoint_3",
    "PatientID_0235_Timepoint_6",
    "PatientID_0236_Timepoint_1",
    "PatientID_0236_Timepoint_3",
    "PatientID_0236_Timepoint_6",
    "PatientID_0268_Timepoint_1",
    "PatientID_0268_Timepoint_3",
    "PatientID_0268_Timepoint_6",
    "PatientID_0274_Timepoint_1",
    "PatientID_0274_Timepoint_3"
  ],
  "outer_test_cases": [
    "PatientID_0003_Timepoint_1",
    "PatientID_0003_Timepoint_2",
    "PatientID_0003_Timepoint_5",
    "PatientID_0009_Timepoint_2",
    "PatientID_0020_Timepoint_1",
    "PatientID_0020_Timepoint_2",
    "PatientID_0020_Timepoint_3",
    "PatientID_0020_Timepoint_4",
    "PatientID_0023_Timepoint_2",
    "PatientID_0035_Timepoint_1",
    "PatientID_0035_Timepoint_2",
    "PatientID_0035_Timepoint_3",
    "PatientID_0046_Timepoint_1",
    "PatientID_0052_Timepoint_1",
    "PatientID_0052_Timepoint_2",
    "PatientID_0057_Timepoint_1",
    "PatientID_0059_Timepoint_1",
    "PatientID_0059_Timepoint_2",
    "PatientID_0059_Timepoint_3",
    "PatientID_0062_Timepoint_1",
    "PatientID_0062_Timepoint_2",
    "PatientID_0065_Timepoint_1",
    "PatientID_0065_Timepoint_2",
    "PatientID_0065_Timepoint_3",
    "PatientID_0065_Timepoint_4",
    "PatientID_0065_Timepoint_5",
    "PatientID_0070_Timepoint_1",
    "PatientID_0070_Timepoint_2",
    "PatientID_0070_Timepoint_3",
    "PatientID_0070_Timepoint_4",
    "PatientID_0072_Timepoint_1",
    "PatientID_0072_Timepoint_2",
    "PatientID_0072_Timepoint_3",
    "PatientID_0073_Timepoint_1",
    "PatientID_0073_Timepoint_2",
    "PatientID_0073_Timepoint_3",
    "PatientID_0073_Timepoint_4",
    "PatientID_0088_Timepoint_1",
    "PatientID_0088_Timepoint_2",
    "PatientID_0095_Timepoint_1",
    "PatientID_0095_Timepoint_2",
    "PatientID_0095_Timepoint_3",
    "PatientID_0095_Timepoint_4",
    "PatientID_0095_Timepoint_6",
    "PatientID_0112_Timepoint_1",
    "PatientID_0114_Timepoint_1",
    "PatientID_0114_Timepoint_2",
    "PatientID_0117_Timepoint_1",
    "PatientID_0117_Timepoint_2",
    "PatientID_0117_Timepoint_3",
    "PatientID_0124_Timepoint_1",
    "PatientID_0132_Timepoint_1",
    "PatientID_0132_Timepoint_2",
    "PatientID_0132_Timepoint_3",
    "PatientID_0135_Timepoint_1",
    "PatientID_0135_Timepoint_2",
    "PatientID_0135_Timepoint_3",
    "PatientID_0136_Timepoint_1",
    "PatientID_0136_Timepoint_2",
    "PatientID_0136_Timepoint_3",
    "PatientID_0138_Timepoint_1",
    "PatientID_0138_Timepoint_2",
    "PatientID_0138_Timepoint_3",
    "PatientID_0139_Timepoint_1",
    "PatientID_0140_Timepoint_1",
    "PatientID_0140_Timepoint_2",
    "PatientID_0140_Timepoint_3",
    "PatientID_0162_Timepoint_1",
    "PatientID_0162_Timepoint_2",
    "PatientID_0162_Timepoint_3",
    "PatientID_0162_Timepoint_4",
    "PatientID_0162_Timepoint_5",
    "PatientID_0162_Timepoint_6",
    "PatientID_0189_Timepoint_1",
    "PatientID_0189_Timepoint_2",
    "PatientID_0189_Timepoint_3",
    "PatientID_0189_Timepoint_4",
    "PatientID_0189_Timepoint_5",
    "PatientID_0189_Timepoint_6",
    "PatientID_0191_Timepoint_2",
    "PatientID_0191_Timepoint_3",
    "PatientID_0191_Timepoint_4",
    "PatientID_0192_Timepoint_1",
    "PatientID_0192_Timepoint_2",
    "PatientID_0192_Timepoint_3",
    "PatientID_0202_Timepoint_1",
    "PatientID_0202_Timepoint_2",
    "PatientID_0202_Timepoint_3",
    "PatientID_0203_Timepoint_1",
    "PatientID_0204_Timepoint_1",
    "PatientID_0204_Timepoint_2",
    "PatientID_0204_Timepoint_3",
    "PatientID_0204_Timepoint_4",
    "PatientID_0204_Timepoint_5",
    "PatientID_0204_Timepoint_6",
    "PatientID_0206_Timepoint_1",
    "PatientID_0211_Timepoint_1",
    "PatientID_0211_Timepoint_2",
    "PatientID_0211_Timepoint_3",
    "PatientID_0215_Timepoint_1",
    "PatientID_0215_Timepoint_2",
    "PatientID_0215_Timepoint_3",
    "PatientID_0215_Timepoint_4",
    "PatientID_0215_Timepoint_5",
    "PatientID_0215_Timepoint_6",
    "PatientID_0244_Timepoint_1",
    "PatientID_0251_Timepoint_1",
    "PatientID_0251_Timepoint_2",
    "PatientID_0251_Timepoint_3",
    "PatientID_0252_Timepoint_1",
    "PatientID_0252_Timepoint_2",
    "PatientID_0252_Timepoint_3",
    "PatientID_0254_Timepoint_1",
    "PatientID_0254_Timepoint_2",
    "PatientID_0254_Timepoint_3",
    "PatientID_0254_Timepoint_4",
    "PatientID_0271_Timepoint_1",
    "PatientID_0271_Timepoint_3",
    "PatientID_0271_Timepoint_6"
  ],
  "role_balance_diagnostics": {
    "fit_train": {
      "patients": 130,
      "scans": 381,
      "label_present_scans": {
        "1": 202,
        "2": 378,
        "3": 351,
        "4": 334
      },
      "label_absent_scans": {
        "1": 179,
        "2": 3,
        "3": 30,
        "4": 47
      }
    },
    "tuning": {
      "patients": 32,
      "scans": 94,
      "label_present_scans": {
        "1": 50,
        "2": 94,
        "3": 86,
        "4": 82
      },
      "label_absent_scans": {
        "1": 44,
        "2": 0,
        "3": 8,
        "4": 12
      }
    },
    "outer_test": {
      "patients": 41,
      "scans": 119,
      "label_present_scans": {
        "1": 63,
        "2": 119,
        "3": 109,
        "4": 104
      },
      "label_absent_scans": {
        "1": 56,
        "2": 0,
        "3": 10,
        "4": 15
      }
    }
  },
  "missing_masks_excluded": [
    "PatientID_0187_Timepoint_3",
    "PatientID_0191_Timepoint_1"
  ]
}""")
fit_train_patients = set(split_record['fit_train_patients'])
tuning_patients = set(split_record['tuning_patients'])
outer_test_patients = set(split_record['outer_test_patients'])
fit_train_case_ids = split_record['fit_train_cases']
tuning_case_ids = split_record['tuning_cases']
outer_test_case_ids = split_record['outer_test_cases']
modeling_case_ids = fit_train_case_ids + tuning_case_ids
modeling_cases = [case_by_id[case_id] for case_id in modeling_case_ids]
outer_test_cases = [case_by_id[case_id] for case_id in outer_test_case_ids]

assert len(patients) == 203
assert missing_masks == split_record['missing_masks_excluded']
assert len(fit_train_case_ids) == EXPECTED_FIT_TRAIN_CASES
assert len(fit_train_patients) == EXPECTED_FIT_TRAIN_PATIENTS
assert len(tuning_case_ids) == EXPECTED_TUNING_CASES
assert len(tuning_patients) == EXPECTED_TUNING_PATIENTS
assert len(outer_test_case_ids) == EXPECTED_OUTER_TEST_CASES
assert len(outer_test_patients) == EXPECTED_OUTER_TEST_PATIENTS
assert fit_train_patients.isdisjoint(tuning_patients)
assert fit_train_patients.isdisjoint(outer_test_patients)
assert tuning_patients.isdisjoint(outer_test_patients)
assert fit_train_patients | tuning_patients | outer_test_patients == set(patients)
assert set(modeling_case_ids).isdisjoint(outer_test_case_ids)
assert set(modeling_case_ids) | set(outer_test_case_ids) == set(case_by_id)

split_text = json.dumps(split_record, indent=2)
split_sha256 = hashlib.sha256(split_text.encode('utf-8')).hexdigest()
assert split_sha256 == EXPECTED_SPLIT_SHA256, (
    f'Fold checksum mismatch: {split_sha256}. Stop: this is not the prespecified split.'
)
(output_root / 'patient_split.json').write_text(split_text, encoding='utf-8')

for case_id in outer_test_case_ids:
    destination = truth_dir / f'{case_id}.nii.gz'
    if not destination.exists():
        shutil.copy2(case_by_id[case_id]['mask'], destination)
    for image_path, (_, channel) in zip(case_by_id[case_id]['images'], modalities):
        link = outer_test_input_dir / f'{case_id}_{channel}.nii.gz'
        if not link.exists() and not link.is_symlink():
            link.symlink_to(image_path)

print('Run:', RUN_ID, '/ fold:', CV_FOLD, '/ training seed:', TRAINING_SEED)
print(f'Total: {len(cases)} cases / {len(patients)} patients')
print(f'Fit training: {len(fit_train_case_ids)} cases / {len(fit_train_patients)} patients')
print(f'Inner tuning: {len(tuning_case_ids)} cases / {len(tuning_patients)} patients')
print(f'Untouched outer test: {len(outer_test_case_ids)} cases / {len(outer_test_patients)} patients')
print('Split SHA-256:', split_sha256)
print('Excluded missing masks:', missing_masks)


## Create nnU-Net raw data from fit-training and inner-tuning patients only


In [ ]:
dataset_dir = raw_root / DATASET_NAME
images_tr = dataset_dir / 'imagesTr'
labels_tr = dataset_dir / 'labelsTr'
if dataset_dir.exists():
    shutil.rmtree(dataset_dir)
images_tr.mkdir(parents=True, exist_ok=True)
labels_tr.mkdir(parents=True, exist_ok=True)

def ensure_link(source_path, destination):
    if destination.exists() or destination.is_symlink():
        return
    destination.symlink_to(source_path)

for number, case in enumerate(modeling_cases, 1):
    for image_path, (_, channel) in zip(case['images'], modalities):
        ensure_link(image_path, images_tr / f'{case["id"]}_{channel}.nii.gz')
    ensure_link(case['mask'], labels_tr / f'{case["id"]}.nii.gz')
    if number % 100 == 0 or number == len(modeling_cases):
        print(f'Linked {number}/{len(modeling_cases)} cases')

dataset_json = {
    'channel_names': {'0': 'T1', '1': 'T1CE', '2': 'T2', '3': 'FLAIR'},
    'labels': {
        'background': 0,
        'non_enhancing_tumor_core': 1,
        'flair_hyperintensity_edema': 2,
        'enhancing_tissue': 3,
        'resection_cavity': 4,
    },
    'numTraining': len(modeling_cases),
    'file_ending': '.nii.gz',
}
(dataset_dir / 'dataset.json').write_text(json.dumps(dataset_json, indent=2))
(output_root / 'dataset.json').write_text(json.dumps(dataset_json, indent=2))
print('nnU-Net raw dataset ready:', dataset_dir)


## Fingerprint, plan, and preprocess modeling data only (`3d_fullres`)


In [ ]:
preprocessed_dataset = preprocessed_root / DATASET_NAME
plans_path = preprocessed_dataset / 'nnUNetPlans.json'


def processed_case_status():
    if not plans_path.exists():
        return False, None, None, {}
    try:
        plans = json.loads(plans_path.read_text())
        identifier = plans['configurations']['3d_fullres']['data_identifier']
    except Exception:
        return False, None, None, {}
    configuration_dir = preprocessed_dataset / identifier
    expected = set(modeling_case_ids)
    b2nd_data = {
        path.name.removesuffix('.b2nd')
        for path in configuration_dir.glob('*.b2nd')
        if not path.name.endswith('_seg.b2nd') and path.stat().st_size > 0
    }
    b2nd_seg = {
        path.name.removesuffix('_seg.b2nd')
        for path in configuration_dir.glob('*_seg.b2nd')
        if path.stat().st_size > 0
    }
    properties = {
        path.stem for path in configuration_dir.glob('*.pkl')
        if path.stat().st_size > 0
    }
    npz_data = {
        path.stem for path in configuration_dir.glob('*.npz')
        if path.stat().st_size > 0
    }
    if b2nd_data or b2nd_seg:
        inventory = {
            'data': b2nd_data,
            'segmentation': b2nd_seg,
            'properties': properties,
        }
        complete = all(values == expected for values in inventory.values())
        return complete, configuration_dir, 'b2nd', inventory
    inventory = {'data': npz_data, 'properties': properties}
    complete = all(values == expected for values in inventory.values())
    return complete, configuration_dir, 'npz', inventory


preprocessing_complete, configuration_dir, storage_format, processed_inventory = (
    processed_case_status()
)
if not preprocessing_complete:
    # A count alone is unsafe: interruption can leave a final data file without
    # its segmentation/properties, and another fold may have the same case
    # count. Rebuild unless every exact case has every non-empty artifact.
    if preprocessed_dataset.exists():
        shutil.rmtree(preprocessed_dataset)

    # nnU-Net normally copies these after all cases are preprocessed. Copy them
    # first because completed source MRI files are reclaimed during preprocessing.
    gt_segmentations = preprocessed_dataset / 'gt_segmentations'
    gt_segmentations.mkdir(parents=True, exist_ok=True)
    for case in modeling_cases:
        shutil.copy2(case['mask'], gt_segmentations / f'{case["id"]}.nii.gz')
    assert {
        path.name.removesuffix('.nii.gz')
        for path in gt_segmentations.glob('*.nii.gz')
    } == set(modeling_case_ids)

    cleanup_stop = threading.Event()
    cleanup_state = {'cases': 0, 'files': 0, 'error': None}

    def reclaim_completed_source_images():
        completed = set()
        while not cleanup_stop.is_set():
            try:
                if plans_path.exists():
                    plans = json.loads(plans_path.read_text())
                    identifier = plans['configurations']['3d_fullres']['data_identifier']
                    output_dir = preprocessed_dataset / identifier
                    for properties_file in output_dir.glob('*.pkl'):
                        case_id = properties_file.stem
                        if case_id in completed or case_id not in set(modeling_case_ids):
                            continue
                        data_file = output_dir / f'{case_id}.b2nd'
                        seg_file = output_dir / f'{case_id}_seg.b2nd'
                        if (
                            not data_file.exists() or not seg_file.exists()
                            or data_file.stat().st_size == 0
                            or seg_file.stat().st_size == 0
                            or properties_file.stat().st_size == 0
                        ):
                            continue
                        removed = 0
                        for image_path in case_by_id[case_id]['images']:
                            if image_path.exists():
                                image_path.unlink()
                                removed += 1
                        completed.add(case_id)
                        cleanup_state['cases'] += 1
                        cleanup_state['files'] += removed
                        if cleanup_state['cases'] % 50 == 0:
                            print(
                                f'Reclaimed source MRI files for '
                                f'{cleanup_state["cases"]} completed cases'
                            )
            except Exception as error:
                cleanup_state['error'] = repr(error)
                return
            cleanup_stop.wait(2)

    cleanup_thread = None
    if extracted_source_root is not None:
        cleanup_thread = threading.Thread(
            target=reclaim_completed_source_images,
            name='source-space-reclaimer',
            daemon=True,
        )
        cleanup_thread.start()

    command = [
        'nnUNetv2_plan_and_preprocess', '-d', str(DATASET_ID),
        '-c', '3d_fullres', '--verify_dataset_integrity',
        '-np', str(PREPROCESS_PROCESSES),
    ]
    print('Running:', ' '.join(command))
    try:
        subprocess.run(command, check=True, env=os.environ.copy())
    finally:
        cleanup_stop.set()
        if cleanup_thread is not None:
            cleanup_thread.join(timeout=10)
    assert cleanup_state['error'] is None, (
        f'Source-space reclaimer failed: {cleanup_state["error"]}'
    )
    if cleanup_thread is not None:
        print(
            f'Reclaimed {cleanup_state["files"]} source MRI files across '
            f'{cleanup_state["cases"]} completed cases during preprocessing'
        )
    preprocessing_complete, configuration_dir, storage_format, processed_inventory = (
        processed_case_status()
    )

assert preprocessing_complete, (
    f'Preprocessing incomplete for the exact {len(modeling_cases)} modeling cases '
    f'in {configuration_dir}: '
    f'{ {key: len(value) for key, value in processed_inventory.items()} }'
)
print('Preprocessing complete:', configuration_dir)
print('Storage format:', storage_format)
disk = shutil.disk_usage(tmp_root)
print(f'Temporary disk: {disk.used/2**30:.1f} GiB used / {disk.total/2**30:.1f} GiB total')


## Install the patient-level fold and reclaim temporary source space

Validation truth masks have already been preserved separately. If the TAR had to be extracted, its four-channel MRI copy is removed now to free roughly 12 GB before training. The uploaded Kaggle Dataset remains untouched.


In [ ]:

manual_split = [{
    'train': sorted(fit_train_case_ids),
    'val': sorted(tuning_case_ids),
}]
split_path = preprocessed_dataset / 'splits_final.json'
split_path.write_text(json.dumps(manual_split, indent=2))
(output_root / 'splits_final.json').write_text(json.dumps(manual_split, indent=2))

assert len(manual_split[0]['train']) == EXPECTED_FIT_TRAIN_CASES
assert len(manual_split[0]['val']) == EXPECTED_TUNING_CASES
assert set(manual_split[0]['train']).isdisjoint(manual_split[0]['val'])
assert set(manual_split[0]['train']) | set(manual_split[0]['val']) == set(modeling_case_ids)
assert set(modeling_case_ids).isdisjoint(outer_test_case_ids)

# Keep the extracted outer-test MRI files until one-time inference is complete.
disk = shutil.disk_usage(tmp_root)
print('Leakage-free inner-tuning split installed:', split_path)
print(
    f'Fold 0 = {EXPECTED_FIT_TRAIN_CASES} fit-train / '
    f'{EXPECTED_TUNING_CASES} inner-tuning cases; '
    f'{EXPECTED_OUTER_TEST_CASES} outer-test cases remain outside nnU-Net planning/training'
)
print(f'Temporary disk: {disk.used/2**30:.1f} / {disk.total/2**30:.1f} GiB')
assert disk.free >= 3 * 2**30, (
    f'Only {disk.free/2**30:.1f} GiB remains for checkpoints and inference; '
    'training is intentionally stopped before risking incomplete outputs.'
)


## Train or resume the convergence-controlled 3D model

The rolling checkpoint is saved after every epoch. With Files-only persistence,
rerunning this notebook can reconstruct preprocessing and resume this unique
run. Training completes at least 100 epochs, ignores gains
smaller than 0.002, then stops once there have been
15 consecutive inner-tuning epochs without a meaningful gain.
Epoch 150 is only an emergency ceiling. The best checkpoint is
then frozen and used once for full-volume outer-test inference.


In [ ]:

trainer_folder = results_root / DATASET_NAME / f'{TRAINER_NAME}__nnUNetPlans__3d_fullres'
fold_dir = trainer_folder / 'fold_0'
final_checkpoint = fold_dir / 'checkpoint_final.pth'
latest_checkpoint = fold_dir / 'checkpoint_latest.pth'
best_checkpoint = fold_dir / 'checkpoint_best.pth'
validation_dir = fold_dir / 'validation'
validation_summary = validation_dir / 'summary.json'
efficiency_path = output_root / 'efficiency.json'
model_complexity_path = output_root / 'model_complexity.json'
preflight_ok_path = output_root / 'preflight_passed.json'


def load_efficiency():
    return json.loads(efficiency_path.read_text()) if efficiency_path.exists() else {
        'run_id': RUN_ID,
        'model': 'nnU-Net v2 3D fullres',
        'gpu': torch.cuda.get_device_name(0),
        'gpu_peak_measurement': 'one-second nvidia-smi samples; maximum total device memory used',
    }


def run_timed(command, timing_key, monitor_gpu=True):
    state = {'peak_gpu_memory_mib': 0, 'samples': 0, 'error': None}
    stop = threading.Event()

    def sample_gpu():
        while not stop.is_set():
            try:
                result = subprocess.run(
                    [
                        'nvidia-smi', '--query-gpu=memory.used',
                        '--format=csv,noheader,nounits',
                    ],
                    check=True, capture_output=True, text=True,
                )
                values = [int(line.strip()) for line in result.stdout.splitlines() if line.strip()]
                if values:
                    state['peak_gpu_memory_mib'] = max(
                        state['peak_gpu_memory_mib'], max(values)
                    )
                    state['samples'] += 1
            except Exception as error:
                state['error'] = repr(error)
                return
            stop.wait(1)

    monitor = None
    if monitor_gpu:
        monitor = threading.Thread(target=sample_gpu, daemon=True)
        monitor.start()
    started = time.perf_counter()
    try:
        subprocess.run(command, check=True, env=os.environ.copy())
    finally:
        elapsed = time.perf_counter() - started
        stop.set()
        if monitor is not None:
            monitor.join(timeout=5)
        efficiency = load_efficiency()
        efficiency[timing_key] = float(efficiency.get(timing_key, 0.0) + elapsed)
        efficiency['peak_observed_gpu_memory_mib'] = max(
            int(efficiency.get('peak_observed_gpu_memory_mib', 0)),
            int(state['peak_gpu_memory_mib']),
        )
        efficiency['gpu_memory_samples'] = int(
            efficiency.get('gpu_memory_samples', 0) + state['samples']
        )
        if state['error'] is not None:
            efficiency['gpu_memory_sampling_error'] = state['error']
        efficiency_path.write_text(json.dumps(efficiency, indent=2))


# Exercise the exact architecture and real modeling data before long training.
# A separate success sentinel prevents a failed/OOM preflight that happened to
# write partial metadata from being skipped on the next Run All.
if not final_checkpoint.exists() and (
    not model_complexity_path.exists() or not preflight_ok_path.exists()
):
    preflight_program = f"""import json
from pathlib import Path
import numpy as np
import torch
from nnunetv2.run.run_training import get_trainer_from_args

trainer = get_trainer_from_args(
    "{DATASET_ID}", "3d_fullres", 0, "{TRAINER_NAME}",
    "nnUNetPlans", False, device=torch.device("cuda")
)
trainer.initialize()
train_loader, tuning_loader = trainer.get_dataloaders()
try:
    total_parameters = sum(parameter.numel() for parameter in trainer.network.parameters())
    trainable_parameters = sum(
        parameter.numel() for parameter in trainer.network.parameters()
        if parameter.requires_grad
    )
    complexity = {{
        "total_parameters": int(total_parameters),
        "trainable_parameters": int(trainable_parameters),
        "planned_patch_size": [int(value) for value in trainer.configuration_manager.patch_size],
        "planned_batch_size": int(trainer.batch_size),
    }}
    trainer.network.train()
    train_result = trainer.train_step(next(train_loader))
    trainer.network.eval()
    with torch.no_grad():
        tuning_result = trainer.validation_step(next(tuning_loader))
    assert np.isfinite(np.asarray(train_result["loss"])).all()
    assert np.isfinite(np.asarray(tuning_result["loss"])).all()
finally:
    for loader in (train_loader, tuning_loader):
        if hasattr(loader, "_finish"):
            loader._finish()
Path(r"{model_complexity_path}").write_text(json.dumps(complexity, indent=2))
Path(r"{preflight_ok_path}").write_text(json.dumps({{
    "status": "passed",
    "real_fit_train_batch": True,
    "real_inner_tuning_batch": True,
}}, indent=2))
print("REAL nnU-Net modeling-data/GPU preflight: passed")
"""
    print('Running disposable nnU-Net real-batch preflight...')
    subprocess.run([sys.executable, '-c', preflight_program], check=True, env=os.environ.copy())

if final_checkpoint.exists() and validation_summary.exists():
    print('Training and inner-tuning checkpoint validation are already complete.')
elif final_checkpoint.exists():
    command = [
        'nnUNetv2_train', str(DATASET_ID), '3d_fullres', '0',
        '-tr', TRAINER_NAME, '--val', '--val_best', '-device', 'cuda',
    ]
    print('Completing interrupted inner-tuning checkpoint validation.')
    run_timed(command, 'tuning_validation_wall_seconds')
else:
    command = [
        'nnUNetv2_train', str(DATASET_ID), '3d_fullres', '0',
        '-tr', TRAINER_NAME, '--val_best', '-device', 'cuda',
    ]
    if latest_checkpoint.exists() or best_checkpoint.exists():
        command.append('--c')
        print('Persistent checkpoint found; resuming training.')
    else:
        print('No checkpoint found; starting training from epoch 1.')
    print('Running:', ' '.join(command))
    run_timed(command, 'training_and_tuning_wall_seconds')

assert preflight_ok_path.exists(), 'Real-data/GPU preflight success record is missing.'
assert model_complexity_path.exists(), 'Model complexity record is missing.'
assert final_checkpoint.exists(), f'Training did not finish: {final_checkpoint}'
assert best_checkpoint.exists(), f'Best tuning checkpoint missing: {best_checkpoint}'
assert validation_summary.exists(), f'Inner-tuning validation did not finish: {validation_summary}'

def outer_predictions_are_complete(require_recorded_timing=True):
    expected = set(outer_test_case_ids)
    predictions = {
        path.name.removesuffix('.nii.gz'): path
        for path in outer_test_prediction_dir.glob('*.nii.gz')
    }
    if set(predictions) != expected:
        return False
    if require_recorded_timing and not (
        float(load_efficiency().get('outer_test_inference_wall_seconds', 0) or 0) > 0
    ):
        return False
    try:
        for case_id, prediction_path in predictions.items():
            truth_image = nib.load(truth_dir / f'{case_id}.nii.gz')
            prediction_image = nib.load(prediction_path)
            if prediction_image.shape != truth_image.shape or not np.allclose(
                prediction_image.affine, truth_image.affine, atol=1e-4
            ):
                return False
            raw_prediction = np.asanyarray(prediction_image.dataobj)
            if (
                not np.issubdtype(raw_prediction.dtype, np.number)
                or not np.all(np.isfinite(raw_prediction))
                or not np.all(raw_prediction == np.rint(raw_prediction))
                or not set(int(value) for value in np.unique(raw_prediction)).issubset(
                    {0, 1, 2, 3, 4}
                )
            ):
                return False
    except Exception as error:
        print('Existing outer prediction validation failed:', repr(error))
        return False
    return True


if not outer_predictions_are_complete(require_recorded_timing=True):
    # Prediction is an atomic, one-time phase. Remove partial/stale outputs and
    # reset its timing so a resumed notebook reports the successful attempt.
    if outer_test_prediction_dir.exists():
        shutil.rmtree(outer_test_prediction_dir)
    outer_test_prediction_dir.mkdir(parents=True, exist_ok=True)
    efficiency = load_efficiency()
    efficiency.pop('outer_test_inference_wall_seconds', None)
    efficiency_path.write_text(json.dumps(efficiency, indent=2))
    command = [
        'nnUNetv2_predict',
        '-i', str(outer_test_input_dir),
        '-o', str(outer_test_prediction_dir),
        '-d', DATASET_NAME,
        '-c', '3d_fullres',
        '-f', '0',
        '-tr', TRAINER_NAME,
        '-p', 'nnUNetPlans',
        '-chk', 'checkpoint_best.pth',
        '-device', 'cuda',
    ]
    print('Running one-time inference on the untouched outer test:', ' '.join(command))
    run_timed(command, 'outer_test_inference_wall_seconds')

assert outer_predictions_are_complete(require_recorded_timing=True), (
    'Outer-test predictions failed exact case, geometry, label, or timing validation.'
)
outer_predictions = list(outer_test_prediction_dir.glob('*.nii.gz'))
assert len(outer_predictions) == EXPECTED_OUTER_TEST_CASES, (
    f'Outer-test prediction count: {len(outer_predictions)} / {EXPECTED_OUTER_TEST_CASES}'
)
assert {path.name.removesuffix('.nii.gz') for path in outer_predictions} == set(outer_test_case_ids)

efficiency = load_efficiency()
if model_complexity_path.exists():
    efficiency.update(json.loads(model_complexity_path.read_text()))
efficiency['best_checkpoint_bytes'] = best_checkpoint.stat().st_size
inference_seconds = efficiency.get('outer_test_inference_wall_seconds')
if inference_seconds:
    efficiency['outer_test_cases'] = EXPECTED_OUTER_TEST_CASES
    efficiency['outer_test_inference_mean_seconds_per_case'] = (
        inference_seconds / EXPECTED_OUTER_TEST_CASES
    )
efficiency_path.write_text(json.dumps(efficiency, indent=2))

if extracted_source_root is not None and extracted_source_root.exists():
    shutil.rmtree(extracted_source_root)
    print('Removed temporary extracted source after untouched outer-test inference.')
print('Training, tuning selection, and untouched outer-test inference complete.')


## Full-volume nnU-Net report

Metrics use the same definitions as the Keras notebook. Empty truth/prediction pairs are excluded for a target; predictions made when the target is absent remain false positives.


In [ ]:
"""Uniform detailed evaluation helpers for all MU-Glioma segmentation runs."""

import hashlib
import math
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import ndimage
from surface_distance import metrics as surface_distance_metrics


SURFACE_DICE_TOLERANCE_MM = 1.0
LESION_CONNECTIVITY = 3  # scipy rank-3/connectivity-3 = 26-connected components

TARGET_DEFINITIONS = {
    "Non-enhancing tumor core": {1},
    "FLAIR hyperintensity / edema": {2},
    "Enhancing tissue": {3},
    "Resection cavity": {4},
    "Tumor-related region (1-3)": {1, 2, 3},
    "All postoperative regions (1-4)": {1, 2, 3, 4},
}

REQUIRED_PER_CASE_COLUMNS = [
    "run_id",
    "model",
    "cv_fold",
    "split_seed",
    "training_seed",
    "case_id",
    "patient_id",
    "target",
    "tp",
    "fp",
    "fn",
    "dice",
    "iou",
    "precision",
    "recall",
    "gt_present",
    "pred_present",
    "truth_voxels",
    "predicted_voxels",
    "voxel_spacing_x_mm",
    "voxel_spacing_y_mm",
    "voxel_spacing_z_mm",
    "surface_distance_defined",
    "hd95_mm",
    "average_surface_distance_mm",
    "average_surface_distance_gt_to_pred_mm",
    "average_surface_distance_pred_to_gt_mm",
    "surface_dice_1mm",
    "reference_lesions",
    "predicted_lesions",
    "detected_reference_lesions",
    "missed_reference_lesions",
    "false_positive_lesions",
    "lesion_sensitivity",
]

METRIC_DEFINITIONS = {
    "version": "mu_glioma_uniform_metrics_v2_fixed_reference_subset",
    "targets": {key: sorted(value) for key, value in TARGET_DEFINITIONS.items()},
    "overlap": {
        "dice": "2TP/(2TP+FP+FN)",
        "iou": "TP/(TP+FP+FN)",
        "precision": "TP/(TP+FP)",
        "recall": "TP/(TP+FN)",
        "both_reference_and_prediction_empty": "Dice/IoU/precision/recall are NaN and excluded from means",
        "reference_empty_prediction_present": "Dice/IoU/precision are 0; recall is NaN",
        "aggregation_subset": "all overlap summaries use the fixed reference-present scans only; reference-absent scans are reported separately",
    },
    "surface": {
        "implementation": "google-deepmind surface-distance 0.1 with physical NIfTI voxel spacing",
        "hd95_mm": "area-weighted symmetric robust Hausdorff distance at the 95th percentile",
        "average_surface_distance_mm": "unweighted mean of the two area-weighted directed average surface distances",
        "surface_dice_tolerance_mm": SURFACE_DICE_TOLERANCE_MM,
        "both_nonempty_required_for_distances": True,
        "one_empty_policy": "HD95/average distances NaN; surface Dice 0",
        "both_empty_policy": "all surface metrics NaN",
    },
    "lesion": {
        "connectivity": "26-connected components in 3D",
        "detection_rule": "a reference component is detected when at least one predicted voxel overlaps it",
        "false_positive_rule": "a predicted component is false-positive when it overlaps no reference voxel",
        "absent_reference_sensitivity": "NaN",
    },
    "absent_reference": {
        "false_positive_scan": "reference target absent and at least one target voxel predicted",
        "confidence_interval": "95% Wilson binomial interval across distinct scans",
        "independence_note": "seed repeats are not counted as new independent patients",
    },
}


def validate_discrete_label_volume(
    array: np.ndarray,
    context: str = "label volume",
    allowed_labels=(0, 1, 2, 3, 4),
) -> np.ndarray:
    """Validate before casting so overflow cannot hide invalid predictions."""
    array = np.asanyarray(array)
    assert np.issubdtype(array.dtype, np.number), (
        f"{context}: non-numeric dtype {array.dtype}"
    )
    assert np.all(np.isfinite(array)), f"{context}: NaN or infinite values"
    assert np.all(array == np.rint(array)), f"{context}: non-integer label values"
    observed = {int(value) for value in np.unique(array)}
    allowed = set(int(value) for value in allowed_labels)
    assert observed.issubset(allowed), (
        f"{context}: unexpected labels {sorted(observed - allowed)}"
    )
    return array.astype(np.uint8, copy=False)


def binary_overlap_metrics(truth: np.ndarray, prediction: np.ndarray) -> dict:
    truth = np.asarray(truth, dtype=bool)
    prediction = np.asarray(prediction, dtype=bool)
    tp = int(np.logical_and(prediction, truth).sum())
    fp = int(np.logical_and(prediction, ~truth).sum())
    fn = int(np.logical_and(~prediction, truth).sum())
    gt_present = bool(truth.any())
    pred_present = bool(prediction.any())
    if not gt_present and not pred_present:
        dice = iou = precision = recall = np.nan
    else:
        dice = (2 * tp) / max(2 * tp + fp + fn, 1)
        iou = tp / max(tp + fp + fn, 1)
        precision = tp / max(tp + fp, 1)
        recall = tp / max(tp + fn, 1) if gt_present else np.nan
    return {
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "dice": dice,
        "iou": iou,
        "precision": precision,
        "recall": recall,
        "gt_present": gt_present,
        "pred_present": pred_present,
        "truth_voxels": int(truth.sum()),
        "predicted_voxels": int(prediction.sum()),
    }


def surface_and_lesion_metrics(
    truth: np.ndarray,
    prediction: np.ndarray,
    spacing_mm,
) -> dict:
    truth = np.asarray(truth, dtype=bool)
    prediction = np.asarray(prediction, dtype=bool)
    spacing_mm = tuple(float(value) for value in spacing_mm)
    gt_present = bool(truth.any())
    pred_present = bool(prediction.any())

    if gt_present and pred_present:
        distances = surface_distance_metrics.compute_surface_distances(
            truth, prediction, spacing_mm=spacing_mm
        )
        directed_asd = surface_distance_metrics.compute_average_surface_distance(
            distances
        )
        hd95_mm = float(
            surface_distance_metrics.compute_robust_hausdorff(distances, 95.0)
        )
        asd_gt_to_pred = float(directed_asd[0])
        asd_pred_to_gt = float(directed_asd[1])
        average_surface_distance = (asd_gt_to_pred + asd_pred_to_gt) / 2.0
        surface_dice = float(
            surface_distance_metrics.compute_surface_dice_at_tolerance(
                distances, tolerance_mm=SURFACE_DICE_TOLERANCE_MM
            )
        )
        surface_distance_defined = True
    elif gt_present or pred_present:
        hd95_mm = np.nan
        asd_gt_to_pred = np.nan
        asd_pred_to_gt = np.nan
        average_surface_distance = np.nan
        surface_dice = 0.0
        surface_distance_defined = False
    else:
        hd95_mm = np.nan
        asd_gt_to_pred = np.nan
        asd_pred_to_gt = np.nan
        average_surface_distance = np.nan
        surface_dice = np.nan
        surface_distance_defined = False

    structure = ndimage.generate_binary_structure(3, LESION_CONNECTIVITY)
    truth_components, reference_lesions = ndimage.label(truth, structure=structure)
    prediction_components, predicted_lesions = ndimage.label(
        prediction, structure=structure
    )
    detected_reference_lesions = sum(
        bool(prediction[truth_components == component].any())
        for component in range(1, reference_lesions + 1)
    )
    false_positive_lesions = sum(
        not bool(truth[prediction_components == component].any())
        for component in range(1, predicted_lesions + 1)
    )
    missed_reference_lesions = reference_lesions - detected_reference_lesions
    lesion_sensitivity = (
        detected_reference_lesions / reference_lesions
        if reference_lesions
        else np.nan
    )
    return {
        "voxel_spacing_x_mm": spacing_mm[0],
        "voxel_spacing_y_mm": spacing_mm[1],
        "voxel_spacing_z_mm": spacing_mm[2],
        "surface_distance_defined": surface_distance_defined,
        "hd95_mm": hd95_mm,
        "average_surface_distance_mm": average_surface_distance,
        "average_surface_distance_gt_to_pred_mm": asd_gt_to_pred,
        "average_surface_distance_pred_to_gt_mm": asd_pred_to_gt,
        "surface_dice_1mm": surface_dice,
        "reference_lesions": int(reference_lesions),
        "predicted_lesions": int(predicted_lesions),
        "detected_reference_lesions": int(detected_reference_lesions),
        "missed_reference_lesions": int(missed_reference_lesions),
        "false_positive_lesions": int(false_positive_lesions),
        "lesion_sensitivity": lesion_sensitivity,
    }


def evaluate_case_targets(
    truth_labels: np.ndarray,
    prediction_labels: np.ndarray,
    spacing_mm,
    identity: dict,
) -> list[dict]:
    rows = []
    for target_name, target_labels in TARGET_DEFINITIONS.items():
        truth = np.isin(truth_labels, list(target_labels))
        prediction = np.isin(prediction_labels, list(target_labels))
        row = dict(identity)
        row["target"] = target_name
        row.update(binary_overlap_metrics(truth, prediction))
        row.update(surface_and_lesion_metrics(truth, prediction, spacing_mm))
        rows.append(row)
    return rows


def metrics_from_counts(tp: int, fp: int, fn: int):
    if tp == 0 and fp == 0 and fn == 0:
        return np.nan, np.nan, np.nan, np.nan
    dice = (2 * tp) / max(2 * tp + fp + fn, 1)
    iou = tp / max(tp + fp + fn, 1)
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1) if (tp + fn) > 0 else np.nan
    return dice, iou, precision, recall


def wilson_interval(successes: int, total: int, z: float = 1.959963984540054):
    if total == 0:
        return np.nan, np.nan
    rate = successes / total
    denominator = 1 + z * z / total
    center = (rate + z * z / (2 * total)) / denominator
    margin = z * math.sqrt(
        rate * (1 - rate) / total + z * z / (4 * total * total)
    ) / denominator
    return max(0.0, center - margin), min(1.0, center + margin)


def build_uniform_summaries(scores: pd.DataFrame):
    missing = [column for column in REQUIRED_PER_CASE_COLUMNS if column not in scores]
    assert not missing, f"Missing required per-case columns: {missing}"

    patient_rows = []
    for (patient_id, target), group in scores.groupby(
        ["patient_id", "target"], sort=True
    ):
        present = group[group.gt_present]
        absent = group[~group.gt_present]
        tp, fp, fn = (
            int(present.tp.sum()),
            int(present.fp.sum()),
            int(present.fn.sum()),
        )
        dice, iou, precision, recall = metrics_from_counts(tp, fp, fn)
        reference_lesions = int(present.reference_lesions.sum())
        detected_lesions = int(present.detected_reference_lesions.sum())
        patient_rows.append(
            {
                "run_id": group.run_id.iloc[0],
                "model": group.model.iloc[0],
                "cv_fold": int(group.cv_fold.iloc[0]),
                "split_seed": int(group.split_seed.iloc[0]),
                "training_seed": int(group.training_seed.iloc[0]),
                "patient_id": patient_id,
                "target": target,
                "scan_count": int(group.case_id.nunique()),
                "reference_present_scan_count": int(present.case_id.nunique()),
                "reference_absent_scan_count": int(absent.case_id.nunique()),
                "tp": tp,
                "fp": fp,
                "fn": fn,
                "pooled_dice": dice,
                "pooled_iou": iou,
                "pooled_precision": precision,
                "pooled_recall": recall,
                "mean_scan_dice": float(present.dice.mean()),
                "median_scan_hd95_mm": float(present.hd95_mm.median()),
                "mean_scan_average_surface_distance_mm": float(
                    present.average_surface_distance_mm.mean()
                ),
                "mean_scan_surface_dice_1mm": float(
                    present.surface_dice_1mm.mean()
                ),
                "reference_lesions": reference_lesions,
                "detected_reference_lesions": detected_lesions,
                "missed_reference_lesions": int(
                    present.missed_reference_lesions.sum()
                ),
                "false_positive_lesions_on_reference_present_scans": int(
                    present.false_positive_lesions.sum()
                ),
                "false_positive_scans_when_reference_absent": int(
                    absent.pred_present.sum()
                ),
                "false_positive_lesions_when_reference_absent": int(
                    absent.false_positive_lesions.sum()
                ),
                "lesion_sensitivity": (
                    detected_lesions / reference_lesions
                    if reference_lesions
                    else np.nan
                ),
            }
        )
    patient_scores = pd.DataFrame(patient_rows)

    summary_rows = []
    absent_rows = []
    for target_name in TARGET_DEFINITIONS:
        subset = scores[scores.target == target_name]
        present = subset[subset.gt_present]
        dice_values = present.dice.dropna()
        q1, q3 = dice_values.quantile([0.25, 0.75])
        reference_lesions = int(present.reference_lesions.sum())
        detected_lesions = int(present.detected_reference_lesions.sum())
        absent = subset[~subset.gt_present]
        false_positive_scans = int(absent.pred_present.sum())
        ci_low, ci_high = wilson_interval(false_positive_scans, len(absent))
        summary_rows.append(
            {
                "run_id": subset.run_id.iloc[0],
                "model": subset.model.iloc[0],
                "cv_fold": int(subset.cv_fold.iloc[0]),
                "split_seed": int(subset.split_seed.iloc[0]),
                "training_seed": int(subset.training_seed.iloc[0]),
                "target": target_name,
                "mean_dice": float(dice_values.mean()),
                "sd_dice": float(dice_values.std(ddof=1)),
                "median_dice": float(dice_values.median()),
                "q1_dice": float(q1),
                "q3_dice": float(q3),
                "mean_iou": float(present.iou.mean()),
                "mean_precision": float(present.precision.mean()),
                "mean_recall": float(present.recall.mean()),
                "median_hd95_mm": float(present.hd95_mm.median()),
                "mean_average_surface_distance_mm": float(
                    present.average_surface_distance_mm.mean()
                ),
                "mean_surface_dice_1mm": float(
                    present.surface_dice_1mm.mean()
                ),
                "reference_lesions": reference_lesions,
                "detected_reference_lesions": detected_lesions,
                "missed_reference_lesions": int(
                    present.missed_reference_lesions.sum()
                ),
                "false_positive_lesions_on_reference_present_scans": int(
                    present.false_positive_lesions.sum()
                ),
                "lesion_sensitivity": (
                    detected_lesions / reference_lesions
                    if reference_lesions
                    else np.nan
                ),
                "validation_scans": int(subset.case_id.nunique()),
                "validation_patients": int(subset.patient_id.nunique()),
                "reference_present_scans": int(present.case_id.nunique()),
                "reference_present_patients": int(present.patient_id.nunique()),
                "absent_reference_scans": int(len(absent)),
                "absent_reference_patients": int(absent.patient_id.nunique()),
                "absent_reference_false_positive_scans": false_positive_scans,
                "absent_reference_false_positive_rate": (
                    false_positive_scans / len(absent) if len(absent) else np.nan
                ),
                "absent_reference_false_positive_rate_ci95_low": ci_low,
                "absent_reference_false_positive_rate_ci95_high": ci_high,
            }
        )
        absent_rows.append(
            {
                "run_id": subset.run_id.iloc[0],
                "model": subset.model.iloc[0],
                "cv_fold": int(subset.cv_fold.iloc[0]),
                "training_seed": int(subset.training_seed.iloc[0]),
                "target": target_name,
                "absent_reference_scans": int(len(absent)),
                "absent_reference_patients": int(absent.patient_id.nunique()),
                "false_positive_scans": false_positive_scans,
                "false_positive_scan_rate": (
                    false_positive_scans / len(absent) if len(absent) else np.nan
                ),
                "false_positive_scan_rate_ci95_low": ci_low,
                "false_positive_scan_rate_ci95_high": ci_high,
                "mean_predicted_voxels_when_absent": float(
                    absent.predicted_voxels.mean()
                ),
                "median_predicted_voxels_when_absent": float(
                    absent.predicted_voxels.median()
                ),
                "false_positive_lesions_when_absent": int(
                    absent.false_positive_lesions.sum()
                ),
            }
        )
    return patient_scores, pd.DataFrame(summary_rows), pd.DataFrame(absent_rows)


def sha256_file(path) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def build_prediction_manifest(prediction_dir, validation_cases: list[dict]):
    case_lookup = {case["id"]: case for case in validation_cases}
    rows = []
    for path in sorted(Path(prediction_dir).glob("*.nii.gz")):
        case_id = path.name.removesuffix(".nii.gz")
        case = case_lookup.get(case_id)
        rows.append(
            {
                "case_id": case_id,
                "patient_id": case["patient"] if case else "",
                "filename": path.name,
                "bytes": path.stat().st_size,
                "sha256": sha256_file(path),
            }
        )
    return pd.DataFrame(rows)


outer_evaluation_cases = [case_by_id[case_id] for case_id in outer_test_case_ids]
per_case = []
missing_predictions = []
for number, case_id in enumerate(outer_test_case_ids, 1):
    prediction_path = outer_test_prediction_dir / f'{case_id}.nii.gz'
    truth_path = truth_dir / f'{case_id}.nii.gz'
    if not prediction_path.exists():
        missing_predictions.append(case_id)
        continue
    truth_image = nib.load(truth_path)
    prediction_image = nib.load(prediction_path)
    truth_labels = validate_discrete_label_volume(
        np.asanyarray(truth_image.dataobj), f'{case_id} reference'
    )
    prediction_labels = validate_discrete_label_volume(
        np.asanyarray(prediction_image.dataobj), f'{case_id} prediction'
    )
    assert prediction_labels.shape == truth_labels.shape, (
        f'Shape mismatch for {case_id}: {prediction_labels.shape} vs {truth_labels.shape}'
    )
    assert np.allclose(prediction_image.affine, truth_image.affine, atol=1e-4), (
        f'Affine mismatch for {case_id}'
    )
    spacing_mm = tuple(float(value) for value in truth_image.header.get_zooms()[:3])
    per_case.extend(
        evaluate_case_targets(
            truth_labels,
            prediction_labels,
            spacing_mm,
            {
                'run_id': RUN_ID,
                'model': 'nnU-Net v2 3D fullres',
                'cv_fold': CV_FOLD,
                'split_seed': SPLIT_SEED,
                'training_seed': TRAINING_SEED,
                'case_id': case_id,
                'patient_id': case_by_id[case_id]['patient'],
            },
        )
    )
    if number % 10 == 0 or number == len(outer_test_case_ids):
        print(f'Evaluated {number}/{len(outer_test_case_ids)} outer-test cases')

assert not missing_predictions, f'Missing predictions: {missing_predictions[:10]}'
scores = pd.DataFrame(per_case)
assert list(scores.columns) == REQUIRED_PER_CASE_COLUMNS
assert scores.case_id.nunique() == EXPECTED_OUTER_TEST_CASES
assert scores.patient_id.nunique() == EXPECTED_OUTER_TEST_PATIENTS
scores.to_csv(output_root / 'outer_test_per_case_metrics.csv', index=False)

patient_scores, summary, absent_reference_summary = build_uniform_summaries(scores)
patient_scores.to_csv(output_root / 'outer_test_per_patient_metrics.csv', index=False)
summary.to_csv(output_root / 'final_summary.csv', index=False)
absent_reference_summary.to_csv(
    output_root / 'absent_reference_false_positive_summary.csv', index=False
)
prediction_manifest = build_prediction_manifest(
    outer_test_prediction_dir, outer_evaluation_cases
)
assert len(prediction_manifest) == EXPECTED_OUTER_TEST_CASES
prediction_manifest.to_csv(
    output_root / 'outer_test_prediction_manifest.csv', index=False
)
(output_root / 'metric_definitions.json').write_text(
    json.dumps(METRIC_DEFINITIONS, indent=2), encoding='utf-8'
)
display(summary)
display(absent_reference_summary)

best_epoch = None
if best_checkpoint.exists():
    checkpoint = torch.load(best_checkpoint, map_location='cpu', weights_only=False)
    best_epoch = int(checkpoint.get('current_epoch', 0))
completed_epochs = None
if final_checkpoint.exists():
    final_state = torch.load(final_checkpoint, map_location='cpu', weights_only=False)
    completed_epochs = int(final_state.get('current_epoch', 0))
convergence_state_path = fold_dir / 'convergence_state.json'
convergence_state = (
    json.loads(convergence_state_path.read_text())
    if convergence_state_path.exists()
    else {}
)

run_summary = {
    'archive_contract_version': 'mu_glioma_detailed_zip_v2_nested',
    'run_id': RUN_ID,
    'architecture': 'nnU-Net v2 3D fullres',
    'cv_fold': CV_FOLD,
    'split_seed': SPLIT_SEED,
    'training_seed': TRAINING_SEED,
    'split_sha256': split_sha256,
    'checkpoint_selection_role': 'patient_grouped_inner_tuning',
    'final_evaluation_role': 'untouched_outer_internal_test',
    'fit_train_cases': EXPECTED_FIT_TRAIN_CASES,
    'fit_train_patients': EXPECTED_FIT_TRAIN_PATIENTS,
    'tuning_cases': EXPECTED_TUNING_CASES,
    'tuning_patients': EXPECTED_TUNING_PATIENTS,
    'outer_test_cases': int(scores.case_id.nunique()),
    'outer_test_patients': int(scores.patient_id.nunique()),
    'trainer': TRAINER_NAME,
    'configuration': '3d_fullres',
    'maximum_epochs': MAX_EPOCHS,
    'minimum_epochs_before_early_stopping': MIN_EPOCHS,
    'completed_epochs': completed_epochs,
    'stop_reason': convergence_state.get('stop_reason'),
    'early_stopping_monitor': 'ema_fg_dice',
    'early_stopping_min_delta': MIN_IMPROVEMENT,
    'early_stopping_patience': EARLY_STOPPING_PATIENCE,
    'best_saved_epoch': best_epoch,
    'best_checkpoint_selected_only_on_inner_tuning': True,
    'best_checkpoint_used_for_outer_test_inference': True,
    'surface_dice_tolerance_mm': SURFACE_DICE_TOLERANCE_MM,
    'lesion_connectivity': '26-connected',
    'prediction_files': len(prediction_manifest),
    'targets': summary.to_dict(orient='records'),
}
(output_root / 'final_summary.json').write_text(json.dumps(run_summary, indent=2))
print('Best saved epoch:', best_epoch)
print('Completed epochs:', completed_epochs)
print('Stop reason:', convergence_state.get('stop_reason'))
print('Detailed reports saved under:', output_root)


## Defer comparisons until all prespecified runs are collected


In [ ]:
print('This notebook intentionally does not compare one fold with the old single-split baseline.')
print('Return MU_Glioma_no30_nnunet_fold2_seed2026_reports.zip; cross-fold and cross-seed statistics are computed after all runs are collected.')


## Training curve and detailed Lightning result archive

The persistent archive includes the best checkpoint, training/convergence records, all untouched outer-test prediction volumes, overlap/surface/lesion metrics, uncertainty summaries, telemetry, checksums, split, configuration, and provenance.


In [ ]:
from PIL import Image
progress_path = fold_dir / 'progress.png'
if progress_path.exists():
    display(Image.open(progress_path))

report_files = [
    output_root / 'patient_split.json',
    output_root / 'dataset.json',
    output_root / 'splits_final.json',
    output_root / 'experiment_config.json',
    output_root / 'environment.json',
    output_root / 'efficiency.json',
    output_root / 'model_complexity.json',
    output_root / 'preflight_passed.json',
    output_root / 'metric_definitions.json',
    output_root / 'outer_test_per_case_metrics.csv',
    output_root / 'outer_test_per_patient_metrics.csv',
    output_root / 'absent_reference_false_positive_summary.csv',
    output_root / 'outer_test_prediction_manifest.csv',
    output_root / 'final_summary.csv',
    output_root / 'final_summary.json',
    validation_summary,
    convergence_state_path,
    progress_path,
    fold_dir / 'debug.json',
    trainer_folder / 'plans.json',
    trainer_folder / 'dataset.json',
    best_checkpoint,
]
required_report_files = [
    output_root / 'patient_split.json',
    output_root / 'experiment_config.json',
    output_root / 'environment.json',
    output_root / 'efficiency.json',
    output_root / 'model_complexity.json',
    output_root / 'preflight_passed.json',
    output_root / 'metric_definitions.json',
    output_root / 'outer_test_per_case_metrics.csv',
    output_root / 'outer_test_per_patient_metrics.csv',
    output_root / 'outer_test_prediction_manifest.csv',
    output_root / 'final_summary.csv',
    output_root / 'final_summary.json',
    convergence_state_path,
    progress_path,
    best_checkpoint,
]
missing_required = [str(path) for path in required_report_files if not path.exists()]
assert not missing_required, f'Required archive artifacts missing: {missing_required}'
archive_entries = []
for path in report_files:
    if path.exists():
        if path == best_checkpoint:
            arcname = Path('model') / path.name
        elif path.parent == fold_dir:
            arcname = Path('training') / path.name
        elif path.parent == validation_dir:
            arcname = Path('training') / 'inner_tuning_summary.json'
        elif path.parent == trainer_folder:
            arcname = Path('model') / path.name
        else:
            arcname = Path('reports') / path.name
        archive_entries.append((path, arcname))
for path in sorted(fold_dir.glob('training_log_*.txt')):
    archive_entries.append((path, Path('training') / path.name))
for path in sorted(outer_test_prediction_dir.glob('*.nii.gz')):
    archive_entries.append((path, Path('outer_test_predictions') / path.name))
for path in sorted(outer_test_prediction_dir.glob('*')):
    if path.is_file() and not path.name.endswith('.nii.gz'):
        archive_entries.append((path, Path('inference_metadata') / path.name))

assert best_checkpoint.exists(), f'Missing best checkpoint: {best_checkpoint}'
assert len(list(outer_test_prediction_dir.glob('*.nii.gz'))) == EXPECTED_OUTER_TEST_CASES
assert len({str(arcname) for _, arcname in archive_entries}) == len(archive_entries)

artifact_manifest = {
    'archive_contract_version': 'mu_glioma_detailed_zip_v2_nested',
    'run_id': RUN_ID,
    'model': 'nnU-Net v2 3D fullres',
    'cv_fold': CV_FOLD,
    'split_seed': SPLIT_SEED,
    'training_seed': TRAINING_SEED,
    'split_sha256': split_sha256,
    'checkpoint_selection_role': 'patient_grouped_inner_tuning',
    'final_evaluation_role': 'untouched_outer_internal_test',
    'prediction_files': EXPECTED_OUTER_TEST_CASES,
    'per_case_metric_rows': int(len(scores)),
    'required_per_case_columns_present': all(
        column in scores.columns for column in REQUIRED_PER_CASE_COLUMNS
    ),
    'includes_all_prediction_volumes': True,
    'includes_best_model_or_checkpoint': True,
    'includes_surface_metrics': True,
    'includes_lesion_metrics': True,
    'includes_absent_reference_uncertainty': True,
}
artifact_manifest_path = output_root / 'artifact_manifest.json'
artifact_manifest_path.write_text(json.dumps(artifact_manifest, indent=2))
archive_entries.append((artifact_manifest_path, Path('reports/artifact_manifest.json')))

checksum_rows = [
    {
        'archive_path': str(arcname),
        'bytes': path.stat().st_size,
        'sha256': sha256_file(path),
    }
    for path, arcname in archive_entries
]
artifact_checksums_path = output_root / 'artifact_checksums.csv'
pd.DataFrame(checksum_rows).to_csv(artifact_checksums_path, index=False)
archive_entries.append((artifact_checksums_path, Path('reports/artifact_checksums.csv')))

import zipfile
report_zip = LIGHTNING_OUTPUT_ROOT / 'MU_Glioma_no30_nnunet_fold2_seed2026_reports.zip'
with zipfile.ZipFile(report_zip, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for path, arcname in archive_entries:
        archive.write(path, arcname=str(arcname))

with zipfile.ZipFile(report_zip) as archive:
    archived_names = archive.namelist()
    assert len(archived_names) == len(set(archived_names))
    assert len([name for name in archived_names if name.startswith('outer_test_predictions/')]) == EXPECTED_OUTER_TEST_CASES
print(json.dumps(artifact_manifest, indent=2))
print(f'Archive size: {report_zip.stat().st_size / 2**30:.2f} GiB')
print('Download and return this complete archive:', report_zip)


## Recovery after a Lightning interruption

Execute the same notebook again in this Studio. Completed preprocessing is verified and nnU-Net resumes from persistent latest/best checkpoints. Do not delete `mu_glioma_nnunet_work/no30` or `mu_glioma_results/MU_Glioma_no30_nnunet_fold2_seed2026` until the final ZIP has been validated.
